In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.integrate import quad, dblquad
from asymptotic import *

# Configuration pour de jolis graphiques
%matplotlib inline
plt.style.use('seaborn-v0_8-muted')

## 2D Interference

In [ ]:
# 2D Interference (Macroscopic Young's Slits) ---
x, y = sp.symbols('x y')

# Distance between sources
d = 1.5
# Phase corresponding to the path difference from two sources at (±d, 0)
# Approximated by parabolas for the analytical example
#phi_int = (x - d)**2 / 2 + y**2 / 2 + (x + d)**2 / 2 + y**2 / 2
# To make the interference interesting, slightly modify the phase
# to create a "lattice" of critical points.
phi_int = sp.sin(x) * sp.cos(y)
amp_int = 1.0

analyzer_int = Analyzer(phi_int, amp_int, [x, y], domain=[(-4, 4), (-4, 4)])

# Search for multiple critical points on a grid
guesses = [np.array([gx, gy]) for gx in np.linspace(-3, 3, 5) for gy in np.linspace(-3, 3, 5)]
points_int = analyzer_int.find_critical_points(guesses)

# Filter unique points
unique_points = []
for p in points_int:
    if not any(np.allclose(p, up, atol=1e-3) for up in unique_points):
        unique_points.append(p)

print(f"Number of critical points found: {len(unique_points)}")

# Visualization of the phase landscape and gradient flow
x_vals = np.linspace(-4, 4, 100)
y_vals = np.linspace(-4, 4, 100)
X, Y = np.meshgrid(x_vals, y_vals)
Z = analyzer_int.func_phase(X, Y)

plt.figure(figsize=(10, 8))
# Background: Phase contours
contour = plt.contourf(X, Y, Z, levels=30, cmap='RdBu_r', alpha=0.8)
plt.colorbar(contour, label="Phase $\phi(x,y)$")

# Overlay critical points
for p in unique_points:
    cp = analyzer_int.analyze_point(p)
    det = np.linalg.det(cp.hessian_matrix)
    if det > 0:
        marker, color = 'o', 'green'  # Extrema
    else:
        marker, color = 's', 'purple' # Saddles

    plt.plot(p[0], p[1], marker=marker, color=color, markersize=10,
             markeredgecolor='white', markeredgewidth=1.5)

# Field lines (Gradient)
U = np.zeros_like(X)
V = np.zeros_like(Y)
# Subsampling for streamplot
X_s, Y_s = X[::4, ::4], Y[::4, ::4]
U_s = np.zeros_like(X_s)
V_s = np.zeros_like(Y_s)

for i in range(X_s.shape[0]):
    for j in range(X_s.shape[1]):
        grad = analyzer_int.func_grad(X_s[i,j], Y_s[i,j])
        U_s[i,j] = grad[0]
        V_s[i,j] = grad[1]

plt.streamplot(X_s, Y_s, U_s, V_s, color='black', linewidth=0.8, density=1.2, arrowsize=1.5)

plt.title("2D Interference Network: Extrema (Green) and Saddles (Purple)", fontsize=15)
plt.xlabel("x", fontsize=13)
plt.ylabel("y", fontsize=13)
plt.axis('equal')
plt.tight_layout()
plt.show()


## Dance of Critical Points (Morse → Airy transition)

In [ ]:
# 1d_dance_critical_points.py


# Symbolic variable
x = sp.symbols('x')

# Parameterized phase: Morse (mu != 0) → Airy (mu = 0)
mu_val = 0.1  # Try values: -0.5, 0.0, 0.5
phi = (x**2/2 + mu_val * x**3 + 0.05 * x**4)
amp = 1.0

# Initialize analyzer
analyzer = Analyzer(phi, amp, [x], domain=[(-3, 3)])
points = analyzer.find_critical_points([np.array([g]) for g in np.linspace(-2, 2, 9)])

print(f"Critical points found ({len(points)}):")
for i, p in enumerate(points):
    cp = analyzer.analyze_point(p)
    print(f"  [{i}] x = {p[0]:.4f} | Type: {cp.singularity_type.value} | φ = {cp.phase_value:.4f}")

# Evaluate asymptotic behavior
evaluator = AsymptoticEvaluator()
lambdas = np.logspace(1, 3, 50)
contributions = []

for lam in lambdas:
    total = 0j
    for p in points:
        cp = analyzer.analyze_point(p)
        res = evaluator.evaluate(cp, lam)
        total += res.total_value
    contributions.append(total)

# Plot magnitude decay
plt.figure(figsize=(10, 6))
plt.loglog(lambdas, np.abs(contributions), 'o-', label='|I(λ)|', linewidth=2)
plt.grid(True, which="both", ls=":", alpha=0.7)
plt.xlabel(r'Frequency parameter $\lambda$ (log scale)', fontsize=12)
plt.ylabel(r'Magnitude $|I(\lambda)|$ (log scale)', fontsize=12)
plt.title(f'Stationary Phase: φ(x) = x²/2 + {mu_val}·x³ + 0.05·x⁴', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

# Plot phase landscape (1D)
x_vals = np.linspace(-2.5, 2.5, 400)
phi_vals = np.array([analyzer.func_phase(xv) for xv in x_vals])

plt.figure(figsize=(10, 4))
plt.plot(x_vals, phi_vals, 'b-', linewidth=2, label=r'$\phi(x)$')
for p in points:
    cp = analyzer.analyze_point(p)
    color = 'red' if cp.singularity_type == 'morse' else 'orange'
    plt.plot(p[0], analyzer.func_phase(p[0]), 'o', color=color, 
             markersize=10, label=f'{cp.singularity_type.value} point')
plt.axhline(0, color='k', ls=':', alpha=0.3)
plt.xlabel('x', fontsize=12)
plt.ylabel(r'$\phi(x)$', fontsize=12)
plt.title('Phase Function Topology (1D)', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Optics of Caustics (Airy diffraction)

In [ ]:
# 1d_caustics_optics.py

x = sp.symbols('x')
a_param = 0.8  # Transverse position relative to caustic

# Canonical Airy phase: φ(x) = x³/3 + a·x  (stationary points when x² = -a)
phi = x**3/3 + a_param * x
amp = 1.0

analyzer = Analyzer(phi, amp, [x], domain=[(-3, 3)])
points = analyzer.find_critical_points([np.array([g]) for g in np.linspace(-2, 2, 7)])

print(f"Critical points for a = {a_param}:")
for p in points:
    cp = analyzer.analyze_point(p)
    print(f"  x = {p[0]:.4f} | Type: {cp.singularity_type.value} | φ = {cp.phase_value:.4f}")

# Exact Airy integral: I(λ) = 2π λ^{-1/3} Ai(λ^{1/3} a)
def exact_airy(lam, a):
    z = (lam * np.abs(a))**(1/3) * np.sign(a)
    Ai_val = airy(z)[0]
    return 2 * np.pi * (lam)**(-1/3) * Ai_val * np.exp(1j * np.pi/6 * np.sign(a))

# Asymptotic evaluation
evaluator = AsymptoticEvaluator()
lambdas = np.logspace(1, 3, 30)
asymptotic_vals = []
exact_vals = []

for lam in lambdas:
    # Asymptotic sum
    total_asym = 0j
    for p in points:
        cp = analyzer.analyze_point(p)
        res = evaluator.evaluate(cp, lam)
        total_asym += res.total_value
    asymptotic_vals.append(total_asym)
    
    # Exact value
    exact_vals.append(exact_airy(lam, a_param))

# Plot comparison
plt.figure(figsize=(12, 5))

# Magnitude
plt.subplot(1, 2, 1)
plt.loglog(lambdas, np.abs(asymptotic_vals), 'o-', label='Asymptotic', linewidth=2)
plt.loglog(lambdas, np.abs(exact_vals), 's--', label='Exact Airy', linewidth=2)
plt.xlabel(r'$\lambda$', fontsize=12)
plt.ylabel(r'$|I(\lambda)|$', fontsize=12)
plt.title('Magnitude Comparison', fontsize=13)
plt.grid(True, which="both", ls=":", alpha=0.7)
plt.legend()

# Phase
plt.subplot(1, 2, 2)
plt.semilogx(lambdas, np.angle(asymptotic_vals), 'o-', label='Asymptotic', linewidth=2)
plt.semilogx(lambdas, np.angle(exact_vals), 's--', label='Exact Airy', linewidth=2)
plt.xlabel(r'$\lambda$', fontsize=12)
plt.ylabel(r'Phase $\arg I(\lambda)$', fontsize=12)
plt.title('Phase Comparison', fontsize=13)
plt.grid(True, which="both", ls=":", alpha=0.7)
plt.legend()

plt.suptitle(f'Caustic Diffraction: φ(x) = x³/3 + {a_param}·x', fontsize=15)
plt.tight_layout()
plt.show()

# Visualize integrand oscillations at λ=50
lam_test = 50
x_plot = np.linspace(-3, 3, 800)
phi_plot = np.array([analyzer.func_phase(xv) for xv in x_plot])
integrand_real = np.cos(lam_test * phi_plot)  # Real part of exp(iλφ)

plt.figure(figsize=(10, 4))
plt.plot(x_plot, integrand_real, 'b-', alpha=0.7, label=r'$\Re[e^{i\lambda\phi(x)}]$')
for p in points:
    plt.axvline(p[0], color='r', ls='--', alpha=0.6, label='Critical point' if p is points[0] else "")
plt.xlabel('x', fontsize=12)
plt.ylabel('Integrand (real part)', fontsize=12)
plt.title(f'Oscillatory Structure at λ = {lam_test}', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Breath of the Phase (Rhythm and caesura)

In [ ]:
# 1d_breath_phase.py

x = sp.symbols('x')
alpha = 0.7  # Controls oscillation frequency

# Phase with rhythmic structure: φ(x) = sin(x) + αx
phi = sp.sin(x) + alpha * x
amp = 1.0

analyzer = Analyzer(phi, amp, [x], domain=[(-4*np.pi, 4*np.pi)])
points = analyzer.find_critical_points([np.array([g]) for g in np.linspace(-12, 12, 25)])

print(f"Critical points ({len(points)} total):")
for i, p in enumerate(points[:10]):  # Show first 10
    cp = analyzer.analyze_point(p)
    print(f"  [{i}] x = {p[0]:.4f} | φ = {cp.phase_value:.4f}")

# Plot phase function and its derivative (shows "caesura" locations)
x_vals = np.linspace(-4*np.pi, 4*np.pi, 1000)
phi_vals = np.array([analyzer.func_phase(xv) for xv in x_vals])
dphi_vals = np.array([analyzer.func_grad(xv)[0] for xv in x_vals])

plt.figure(figsize=(12, 6))

# Phase function
plt.subplot(2, 1, 1)
plt.plot(x_vals, phi_vals, 'b-', linewidth=1.5, label=r'$\phi(x)$')
for p in points:
    plt.plot(p[0], analyzer.func_phase(p[0]), 'ro', markersize=6)
plt.ylabel(r'$\phi(x)$', fontsize=12)
plt.title(r'Phase Function $\phi(x) = \sin(x) + \alpha x$ with Critical Points (Caesurae)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()

# Derivative (shows where oscillations slow down)
plt.subplot(2, 1, 2)
plt.plot(x_vals, dphi_vals, 'g-', linewidth=1.5, label=r"$\phi'(x)$")
plt.axhline(0, color='k', ls='--', alpha=0.5)
for p in points:
    plt.axvline(p[0], color='r', ls=':', alpha=0.4)
plt.xlabel('x', fontsize=12)
plt.ylabel(r"$\phi'(x)$", fontsize=12)
plt.title('Derivative: Zero-crossings = Stationary Points (Slow Oscillation Zones)', fontsize=13)
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

# Show integrand structure at moderate λ to visualize "breathing"
lam_test = 20
integrand_real = np.cos(lam_test * phi_vals)

plt.figure(figsize=(12, 4))
plt.plot(x_vals, integrand_real, 'b-', alpha=0.8, linewidth=1)
for p in points:
    plt.axvline(p[0], color='r', ls='--', alpha=0.5)
plt.xlabel('x', fontsize=12)
plt.ylabel(r'$\Re[e^{i\lambda\phi(x)}]$', fontsize=12)
plt.title(f'Oscillatory "Breath": Slow zones near critical points (λ = {lam_test})', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Interference of Two Morse Points

In [ ]:
# 1d_two_morse_interference.py

x = sp.symbols('x')
a_dist = 1.2  # Distance between wells

# Double-well phase: φ(x) = (x² - a²)²/4  → minima at x = ±a, maximum at x=0
phi = (x**2 - a_dist**2)**2 / 4
amp = 1.0

analyzer = Analyzer(phi, amp, [x], domain=[(-3, 3)])
points = analyzer.find_critical_points([np.array([g]) for g in [-2, -0.5, 0.5, 2]])

print("Critical points:")
for p in points:
    cp = analyzer.analyze_point(p)
    typ = "minimum" if cp.hessian_matrix[0,0] > 0 else "maximum"
    print(f"  x = {p[0]:.4f} | Type: {cp.singularity_type.value} ({typ}) | φ = {cp.phase_value:.4f}")

# Evaluate asymptotic contributions separately and combined
evaluator = AsymptoticEvaluator()
lambdas = np.logspace(1, 3, 40)

total_vals = []
contrib1 = []  # Left minimum
contrib2 = []  # Right minimum
contrib_saddle = []  # Central maximum

for lam in lambdas:
    vals = []
    for i, p in enumerate(points):
        cp = analyzer.analyze_point(p)
        res = evaluator.evaluate(cp, lam)
        vals.append(res.total_value)
        if i == 0: contrib1.append(res.total_value)
        elif i == 1: contrib_saddle.append(res.total_value)
        elif i == 2: contrib2.append(res.total_value)
    total_vals.append(sum(vals))

# Plot interference pattern in magnitude
plt.figure(figsize=(10, 6))
plt.loglog(lambdas, np.abs(contrib1), 'o-', label='Left minimum', alpha=0.7)
plt.loglog(lambdas, np.abs(contrib2), 's-', label='Right minimum', alpha=0.7)
plt.loglog(lambdas, np.abs(contrib_saddle), '^-', label='Central saddle', alpha=0.7)
plt.loglog(lambdas, np.abs(total_vals), 'k-', linewidth=2.5, label='Total (interference)', alpha=0.9)
plt.xlabel(r'$\lambda$', fontsize=12)
plt.ylabel(r'$|I(\lambda)|$', fontsize=12)
plt.title(f'Interference of Two Morse Points (a = {a_dist})', fontsize=14)
plt.grid(True, which="both", ls=":", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

# Show phase difference causing interference
phase_diff = np.angle(np.array(contrib1) / np.array(contrib2))
plt.figure(figsize=(10, 4))
plt.semilogx(lambdas, phase_diff, 'o-')
plt.axhline(np.pi, color='r', ls='--', alpha=0.5, label=r'$\pi$ (destructive)')
plt.axhline(0, color='g', ls='--', alpha=0.5, label='0 (constructive)')
plt.xlabel(r'$\lambda$', fontsize=12)
plt.ylabel(r'Phase difference $\Delta\theta$', fontsize=12)
plt.title('Phase Difference Between Minima → Interference Beats', fontsize=14)
plt.grid(True, which="both", ls=":", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

## Boundary of Asymptotic Regime

In [ ]:
# pearcey_convergence_ROBUST.py

# Symboles et phase Pearcey canonique
x, y = sp.symbols('x y')
phi = x**4 / 4 + y**2 / 2
amp = 1.0

# Analyse du point critique
analyzer = Analyzer(phi, amp, [x, y], domain=[(-3, 3), (-3, 3)])
cp = analyzer.analyze_point(analyzer.find_critical_points([np.array([0.0, 0.0])])[0])

print("Pearcey Singularity Analysis")
print("="*60)
print(f"Phase: φ(x,y) = x⁴/4 + y²/2")
print(f"Critical point: ({cp.position[0]:.4f}, {cp.position[1]:.4f})")
print(f"Type: {cp.singularity_type.value}")
print(f"Quartic coeff: {cp.canonical_coefficients['quartic']:.4f}")
print(f"Transverse coeff: {cp.canonical_coefficients['quadratic_transverse']:.4f}")
print("="*60 + "\n")

# ====================================================================================
# MÉTHODE 1 : Valeur exacte factorisée (référence absolue)
# ====================================================================================
def exact_pearcey(lam):
    """
    Exact value via factorization:
    I(λ) = [∫exp(iλx⁴/4)dx] × [∫exp(iλy²/2)dy]
         = [½Γ(¼)(4/λ)^¼ e^{iπ/8}] × [√(2π/λ) e^{iπ/4}]
         = ½Γ(¼)√(2π)·4^{¼}·λ^{-¾} e^{i3π/8}
    """
    coeff = 0.5 * gamma(0.25) * np.sqrt(2*np.pi) * (4**0.25)
    phase = 3 * np.pi / 8  # 67.5 degrees
    return coeff * lam**(-0.75) * np.exp(1j * phase)

# ====================================================================================
# MÉTHODE 2 : Intégration numérique 1D factorisée (robuste)
# ====================================================================================
def numerical_factorized(lam, Lx=5.0, Ly=5.0, Nx=20001, Ny=20001):
    """
    Compute I(λ) = I_x(λ) × I_y(λ) with high-resolution 1D quadrature.
    Avoids 2D aliasing by separating variables.
    """
    # Intégrale quartique en x
    x_grid = np.linspace(-Lx, Lx, Nx)
    phase_x = lam * x_grid**4 / 4
    integrand_x = np.exp(1j * phase_x)
    I_x = np.trapezoid(integrand_x, x_grid)
    
    # Intégrale gaussienne en y
    y_grid = np.linspace(-Ly, Ly, Ny)
    phase_y = lam * y_grid**2 / 2
    integrand_y = np.exp(1j * phase_y)
    I_y = np.trapezoid(integrand_y, y_grid)
    
    return I_x * I_y

# ====================================================================================
# MÉTHODE 3 : Asymptotique via asymptotic.py
# ====================================================================================
evaluator = AsymptoticEvaluator()

# ====================================================================================
# Évaluation comparative
# ====================================================================================
lambdas = np.logspace(0.7, 3.0, 20)  # λ de 5 à 1000

exact_vals = []
numerical_vals = []
asymptotic_vals = []

print(f"{'λ':>8} | {'Exact |I|':>12} | {'Numerical |I|':>15} | {'Asymptotic |I|':>16} | {'Err_num (%)':>10} | {'Err_asy (%)':>10}")
print("-"*95)

for lam in lambdas:
    # Valeur exacte
    exact_val = exact_pearcey(lam)
    exact_vals.append(exact_val)
    
    # Valeur numérique factorisée (robuste)
    num_val = numerical_factorized(
        lam, 
        Lx=8*lam**(-0.25),  # Domaine adaptatif
        Ly=8*lam**(-0.5),
        Nx=20001,           # Résolution extrême (20k points)
        Ny=20001
    )
    numerical_vals.append(num_val)
    
    # Valeur asymptotique
    asym_val = evaluator.evaluate(cp, lam).total_value
    asymptotic_vals.append(asym_val)
    
    # Erreurs relatives
    err_num = np.abs((num_val - exact_val) / exact_val) * 100
    err_asy = np.abs((asym_val - exact_val) / exact_val) * 100
    
    print(f"{lam:8.1f} | {np.abs(exact_val):12.6f} | {np.abs(num_val):15.6f} | {np.abs(asym_val):16.6f} | {err_num:10.2f} | {err_asy:10.2f}")

# ====================================================================================
# Visualisation de la convergence
# ====================================================================================
exact_mags = np.abs(exact_vals)
numerical_mags = np.abs(numerical_vals)
asymptotic_mags = np.abs(asymptotic_vals)

rel_err_num = np.abs((np.array(numerical_vals) - np.array(exact_vals)) / np.array(exact_vals))
rel_err_asy = np.abs((np.array(asymptotic_vals) - np.array(exact_vals)) / np.array(exact_vals))

plt.figure(figsize=(14, 5))

# Magnitudes
plt.subplot(1, 2, 1)
plt.loglog(lambdas, exact_mags, 'k-', linewidth=3, label='Exact (factorized)', alpha=0.9)
plt.loglog(lambdas, numerical_mags, 'bo', markersize=6, label='Numerical (1D factorized)', alpha=0.7)
plt.loglog(lambdas, asymptotic_mags, 'r--', linewidth=2.5, label='Asymptotic (asymptotic.py)', alpha=0.9)
plt.xlabel(r'Frequency parameter $\lambda$', fontsize=13)
plt.ylabel(r'Magnitude $|I(\lambda)|$', fontsize=13)
plt.title('Pearcey Integral: Exact vs Numerical vs Asymptotic', fontsize=14, fontweight='bold')
plt.grid(True, which="both", ls=":", alpha=0.7)
plt.legend(fontsize=10)

# Erreurs relatives
plt.subplot(1, 2, 2)
plt.loglog(lambdas, rel_err_num, 'bo-', linewidth=2, markersize=6, label='Numerical error', alpha=0.8)
plt.loglog(lambdas, rel_err_asy, 'r^-', linewidth=2.5, markersize=8, label='Asymptotic error', alpha=0.9)
plt.loglog(lambdas, 0.5 * lambdas**-1, 'b--', linewidth=2, alpha=0.6, label=r'Theoretical $O(\lambda^{-1})$')
plt.axhline(0.01, color='green', ls=':', linewidth=2, alpha=0.7, label='1% error')
plt.xlabel(r'Frequency parameter $\lambda$', fontsize=13)
plt.ylabel('Relative error', fontsize=13)
plt.title('Convergence Rates', fontsize=14, fontweight='bold')
plt.grid(True, which="both", ls=":", alpha=0.7)
plt.legend(fontsize=10)

plt.suptitle(r'Pearcey Catastrophe: $\phi(x,y) = \frac{x^4}{4} + \frac{y^2}{2}$', 
             fontsize=16, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

# ====================================================================================
# Validation détaillée à λ=100
# ====================================================================================
lam_test = 100.0
exact_test = exact_pearcey(lam_test)
num_test = numerical_factorized(lam_test, Lx=8*lam_test**(-0.25), Ly=8*lam_test**(-0.5), Nx=40001, Ny=40001)
asym_test = evaluator.evaluate(cp, lam_test).total_value

print("\n" + "="*70)
print(f"DETAILED VALIDATION AT λ = {lam_test}")
print("="*70)
print(f"Exact value      : {exact_test:.10f}  |  |I| = {np.abs(exact_test):.6f}")
print(f"Numerical (1D)   : {num_test:.10f}  |  |I| = {np.abs(num_test):.6f}  |  error = {np.abs((num_test-exact_test)/exact_test)*100:.3f}%")
print(f"Asymptotic       : {asym_test:.10f}  |  |I| = {np.abs(asym_test):.6f}  |  error = {np.abs((asym_test-exact_test)/exact_test)*100:.3f}%")
print("="*70)

# ====================================================================================
# Pourquoi la méthode 2D échoue : visualisation de l'aliasing
# ====================================================================================
lam_alias = 50.0
Lx_alias = 4.0
x_fine = np.linspace(-Lx_alias, Lx_alias, 20001)   # dx ≈ 0.0004 (résolution correcte)
x_coarse = np.linspace(-Lx_alias, Lx_alias, 401)   # dx ≈ 0.02 (votre résolution)

phase_fine = lam_alias * x_fine**4 / 4
phase_coarse = lam_alias * x_coarse**4 / 4

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(x_fine, np.cos(phase_fine), 'b-', linewidth=1, alpha=0.7, label='Fine grid (20k pts)')
plt.plot(x_coarse, np.cos(phase_coarse), 'ro', markersize=3, alpha=0.5, label='Coarse grid (400 pts)')
plt.xlim(-1.5, 1.5)
plt.xlabel('x', fontsize=11)
plt.ylabel(r'$\cos(\lambda x^4/4)$', fontsize=11)
plt.title(f'Oscillations at λ={lam_alias} (zoom near x=1)', fontsize=12, fontweight='bold')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# Spectre de fréquence locale
x_local = np.linspace(0.1, 2.0, 200)
freq_local = lam_alias * x_local**3 / (2*np.pi)  # cycles/unité
plt.plot(x_local, freq_local, 'g-', linewidth=2.5)
plt.axhline(1/(2*0.02), color='r', ls='--', alpha=0.7, label='Nyquist limit (dx=0.02)')
plt.axhline(1/(2*0.0004), color='b', ls='--', alpha=0.7, label='Nyquist limit (dx=0.0004)')
plt.xlabel('Position x', fontsize=11)
plt.ylabel('Local frequency (cycles/unit)', fontsize=11)
plt.title('Local Frequency vs Nyquist Limits', fontsize=12, fontweight='bold')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)

plt.suptitle('Why 2D Grid Integration Fails: Aliasing at High Frequencies', 
             fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)
print("1. The asymptotic formula in asymptotic.py is CORRECT")
print("   (validated against exact factorized solution)")
print("")
print("2. Naive 2D grid integration FAILS due to:")
print("   • Insufficient resolution for rapid oscillations (aliasing)")
print("   • Local frequency grows as λ·x³ → requires dx << λ^{-1/3}")
print("   • At λ=100, need dx < 0.01 near x=1 (your dx≈0.016 → borderline)")
print("")
print("3. ROBUST numerical strategy:")
print("   • Factorize when possible (separable phases)")
print("   • Use 1D quadrature with extreme resolution (20k+ points)")
print("   • For non-separable phases: use Levin/Filon methods or")
print("     complex contour deformation")
print("="*70)

## Mythopoetic Singularity (Archetypal journey)

In [ ]:
# 1d_mythopoetic_singularity.py
import matplotlib.patches as mpatches


x = sp.symbols('x')

# Airy canonical form with bifurcation parameter μ
mu_values = [-1.5, -0.5, 0.0, 0.5, 1.5]  # Journey stages
phi_template = x**3/3 - sp.Symbol('mu') * x  # φ(x) = x³/3 - μx

fig, axes = plt.subplots(len(mu_values), 2, figsize=(14, 16))
fig.suptitle('Mythopoetic Journey Through a Catastrophe Singularity\n'
             r'$\phi(x) = \frac{x^3}{3} - \mu x$', 
             fontsize=16, y=0.995)

stages = [
    "I. Duality (μ < 0): Two critical points",
    "II. Approach (μ → 0⁻): Points converge",
    "III. Crisis (μ = 0): Fusion into singularity",
    "IV. Transcendence (μ > 0): No real critical points",
    "V. Resolution (μ ≫ 0): Pure oscillation"
]

for idx, mu_val in enumerate(mu_values):
    # Create phase for this μ
    phi = x**3/3 - mu_val * x
    analyzer = Analyzer(phi, 1.0, [x], domain=[(-3, 3)])
    points = analyzer.find_critical_points([np.array([g]) for g in np.linspace(-2, 2, 9)])
    
    # Plot phase function
    ax_phase = axes[idx, 0]
    x_vals = np.linspace(-2.5, 2.5, 400)
    phi_vals = np.array([analyzer.func_phase(xv) for xv in x_vals])
    ax_phase.plot(x_vals, phi_vals, 'b-', linewidth=2.5, alpha=0.8)
    
    # Mark critical points
    for p in points:
        cp = analyzer.analyze_point(p)
        color = 'red' if cp.singularity_type == 'morse' else 'orange'
        ax_phase.plot(p[0], analyzer.func_phase(p[0]), 'o', color=color, 
                     markersize=12, zorder=5)
    
    ax_phase.axhline(0, color='k', ls=':', alpha=0.3)
    ax_phase.set_ylabel(r'$\phi(x)$', fontsize=11)
    ax_phase.set_title(f'{stages[idx]}\nμ = {mu_val}', fontsize=12, fontweight='bold')
    ax_phase.grid(True, alpha=0.3)
    if idx == len(mu_values)-1:
        ax_phase.set_xlabel('x', fontsize=11)
    
    # Plot integrand oscillations at λ=30
    ax_int = axes[idx, 1]
    lam_test = 30
    integrand_real = np.cos(lam_test * phi_vals)
    ax_int.plot(x_vals, integrand_real, 'purple', alpha=0.85, linewidth=1.5)
    
    # Shade stationary zones
    for p in points:
        ax_int.axvspan(p[0]-0.3, p[0]+0.3, color='yellow', alpha=0.4, zorder=0)
    
    ax_int.set_ylabel('Re[integrand]', fontsize=11)
    ax_int.set_title(f'Oscillatory Structure (λ = {lam_test})', fontsize=12)
    ax_int.grid(True, alpha=0.3)
    if idx == len(mu_values)-1:
        ax_int.set_xlabel('x', fontsize=11)
    
    # Add narrative annotation
    if mu_val < 0:
        n_points = len(points)
        ax_phase.text(0.05, 0.9, f'{n_points} real critical points', 
                     transform=ax_phase.transAxes, fontsize=10,
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    elif mu_val == 0:
        ax_phase.text(0.05, 0.9, 'SINGULARITY: degenerate critical point', 
                     transform=ax_phase.transAxes, fontsize=10, color='red',
                     bbox=dict(boxstyle='round', facecolor='pink', alpha=0.9))
    else:
        ax_phase.text(0.05, 0.9, 'No real critical points → pure oscillation', 
                     transform=ax_phase.transAxes, fontsize=10, color='gray',
                     bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

plt.tight_layout(rect=[0, 0.01, 1, 0.98])
plt.show()

# Final plot: magnitude vs μ at fixed λ
lam_fixed = 50
mu_range = np.linspace(-2, 2, 80)
magnitudes = []

for mu_val in mu_range:
    phi = x**3/3 - mu_val * x
    analyzer = Analyzer(phi, 1.0, [x], domain=[(-3, 3)])
    points = analyzer.find_critical_points([np.array([g]) for g in np.linspace(-2, 2, 9)])
    
    evaluator = AsymptoticEvaluator()
    total = 0j
    for p in points:
        cp = analyzer.analyze_point(p)
        res = evaluator.evaluate(cp, lam_fixed)
        total += res.total_value
    magnitudes.append(np.abs(total))

plt.figure(figsize=(10, 5))
plt.plot(mu_range, magnitudes, 'b-', linewidth=2.5)
plt.axvline(0, color='r', ls='--', alpha=0.7, label='Singularity (μ=0)')
plt.xlabel(r'Bifurcation parameter $\mu$', fontsize=13)
plt.ylabel(r'$|I(\lambda={})|$'.format(lam_fixed), fontsize=13)
plt.title('Archetypal Journey: Magnitude Through the Singularity', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Topology of Stationary Valleys

In [ ]:
# 2d_stationary_valleys.py

x, y = sp.symbols('x y')

# Double-well potential in x, harmonic in y
phi = x**4/4 - x**2/2 + y**2/2
amp = 1.0

analyzer = Analyzer(phi, amp, [x, y], domain=[(-2, 2), (-2, 2)])
points = analyzer.find_critical_points([
    np.array([-1.0, 0.0]),
    np.array([0.0, 0.0]),
    np.array([1.0, 0.0]),
    np.array([0.5, 0.5])
])

print("Critical points analysis:")
for p in points:
    cp = analyzer.analyze_point(p)
    typ = "min" if cp.hessian_matrix[0,0] > 0 and cp.hessian_matrix[1,1] > 0 else \
          "saddle" if np.linalg.det(cp.hessian_matrix) < 0 else "max"
    print(f"  ({p[0]:.3f}, {p[1]:.3f}) | Type: {cp.singularity_type.value} ({typ}) | φ = {cp.phase_value:.3f}")

# Create visualizer
viz = StationaryPhaseVisualizer(analyzer)

# 1. Phase landscape with critical points
viz.plot_phase_landscape(
    [analyzer.analyze_point(p) for p in points],
    bounds=((-1.8, 1.8), (-1.8, 1.8)),
    points_per_axis=150
)

# 2. Gradient field + critical points (custom plot)
x_vals = np.linspace(-1.8, 1.8, 25)
y_vals = np.linspace(-1.8, 1.8, 25)
X, Y = np.meshgrid(x_vals, y_vals)

# Compute gradient components
U = np.zeros_like(X)
V = np.zeros_like(Y)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        grad = np.array(analyzer.func_grad(X[i,j], Y[i,j]))
        U[i,j] = -grad[0]  # Negative for "flow toward minima"
        V[i,j] = -grad[1]

plt.figure(figsize=(10, 8))
plt.streamplot(X, Y, U, V, color='lightblue', density=1.2, linewidth=1, arrowsize=1.2)
for p in points:
    cp = analyzer.analyze_point(p)
    if cp.singularity_type == 'morse':
        if np.linalg.det(cp.hessian_matrix) > 0:  # Minimum
            plt.plot(p[0], p[1], 'go', markersize=12, label='Minimum' if p is points[0] else "")
        else:  # Saddle
            plt.plot(p[0], p[1], 'rs', markersize=12, label='Saddle' if p is points[1] else "")
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Gradient Flow Field: Basins of Attraction for Stationary Points', fontsize=14)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.show()

# 3. 3D surface plot of phase function
from mpl_toolkits.mplot3d import Axes3D
x_surf = np.linspace(-1.8, 1.8, 100)
y_surf = np.linspace(-1.8, 1.8, 100)
X_surf, Y_surf = np.meshgrid(x_surf, y_surf)
Z_surf = analyzer.func_phase(X_surf, Y_surf)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X_surf, Y_surf, Z_surf, cmap='viridis', alpha=0.9, 
                       linewidth=0, antialiased=True)
for p in points:
    cp = analyzer.analyze_point(p)
    color = 'red' if np.linalg.det(cp.hessian_matrix) < 0 else 'green'
    ax.scatter(p[0], p[1], analyzer.func_phase(p[0], p[1]), 
              color=color, s=100, edgecolor='white', linewidth=2, zorder=10)
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_zlabel(r'$\phi(x,y)$', fontsize=11)
ax.set_title('3D Phase Topology: Valleys and Saddles', fontsize=14)
plt.colorbar(surf, ax=ax, shrink=0.5, label=r'$\phi$')
plt.tight_layout()
plt.show()

## Cusp Caustic (Deltoid Shape)

In [ ]:
# 2d_cusp_caustic.py

x, y = sp.symbols('x y')

# Canonical cusp catastrophe: φ(x,y) = x³/3 - x·y + y²/2
# Critical set: ∂φ/∂x = x² - y = 0  → parabola y = x²
# Caustic in parameter space has deltoid shape
phi = x**3/3 - x*y + y**2/2
amp = 1.0

analyzer = Analyzer(phi, amp, [x, y], domain=[(-2, 2), (-1, 3)])
points = analyzer.find_critical_points([
    np.array([g, g**2]) for g in np.linspace(-1.5, 1.5, 7)  # Sample along y=x²
])

print("Critical points along the parabola y = x²:")
for p in points:
    cp = analyzer.analyze_point(p)
    print(f"  ({p[0]:.3f}, {p[1]:.3f}) | Type: {cp.singularity_type.value} | det(H) = {cp.hessian_det:.4f}")

# Visualize phase landscape
viz = StationaryPhaseVisualizer(analyzer)
viz.plot_phase_landscape(
    [analyzer.analyze_point(p) for p in points],
    bounds=((-1.8, 1.8), (-0.5, 2.5)),
    points_per_axis=150
)

# Plot the critical manifold (where ∇φ=0) and caustic curve
plt.figure(figsize=(10, 8))

# Critical manifold in (x,y) space: y = x²
x_crit = np.linspace(-1.6, 1.6, 200)
y_crit = x_crit**2
plt.plot(x_crit, y_crit, 'b-', linewidth=2.5, label=r'Critical manifold: $y = x^2$')

# Mark degenerate point (where Hessian det = 0)
degenerate_points = [p for p in points if abs(analyzer.analyze_point(p).hessian_det) < 1e-3]
for p in degenerate_points:
    plt.plot(p[0], p[1], 'ro', markersize=12, label='Degenerate point (cusp)')

# Show gradient magnitude as background
x_bg = np.linspace(-1.8, 1.8, 120)
y_bg = np.linspace(-0.5, 2.5, 120)
X_bg, Y_bg = np.meshgrid(x_bg, y_bg)
grad_mag = np.zeros_like(X_bg)
for i in range(X_bg.shape[0]):
    for j in range(X_bg.shape[1]):
        g = np.array(analyzer.func_grad(X_bg[i,j], Y_bg[i,j]))
        grad_mag[i,j] = np.linalg.norm(g)

plt.contourf(X_bg, Y_bg, grad_mag, levels=30, cmap='Reds_r', alpha=0.4)
plt.colorbar(label=r'$\|\nabla\phi\|$ (small = stationary)')

plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Critical Manifold and Degenerate Cusp Point', fontsize=14)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.show()

# Asymptotic magnitude along a slice through the cusp
evaluator = AsymptoticEvaluator()
y_slice = 0.0  # Horizontal slice through cusp
x_vals = np.linspace(-1.5, 1.5, 60)
lam_test = 80

magnitudes = []
for xv in x_vals:
    # Create phase with offset to simulate parameter variation
    phi_offset = (x**3/3 - x*y + y**2/2).subs({y: y_slice}) + xv * x
    analyzer_slice = Analyzer(phi_offset, 1.0, [x], domain=[(-3, 3)])
    pts = analyzer_slice.find_critical_points([np.array([g]) for g in np.linspace(-2, 2, 9)])
    
    total = 0j
    for p in pts:
        cp = analyzer_slice.analyze_point(p)
        res = evaluator.evaluate(cp, lam_test)
        total += res.total_value
    magnitudes.append(np.abs(total))

plt.figure(figsize=(10, 5))
plt.plot(x_vals, magnitudes, 'm-', linewidth=2.5)
plt.axvline(0, color='r', ls='--', alpha=0.7, label='Cusp location')
plt.xlabel(r'Parameter $x_0$ (transverse to caustic)', fontsize=12)
plt.ylabel(r'$|I(\lambda={})|$'.format(lam_test), fontsize=12)
plt.title('Caustic Intensity Profile: Deltoid Singularity', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Symplectic Dance of Critical Points

In [ ]:
from matplotlib.animation import FuncAnimation
# --- Symbols Configuration ---
q, p = sp.symbols('q p')
alpha_vals = np.linspace(-1.5, 1.5, 60)  # Increased for smoother animation

# --- Data Collection ---
trajectories = {i: [] for i in range(5)} # Increased to 5 to capture all branches

print("Bifurcation analysis in progress...")
for alpha in alpha_vals:
    # Hamiltonian: H(q,p) = p²/2 + V(q)
    # V(q) = cos(q) + alpha * cos(2q)
    H = p**2/2 + sp.cos(q) + alpha * sp.cos(2*q)

    # Using your Analyzer
    analyzer = Analyzer(H, 1.0, [q, p], domain=[(-np.pi, np.pi), (-1, 1)])

    # Strategy for finding critical points on the p=0 axis
    guesses = [np.array([g, 0.0]) for g in np.linspace(-np.pi, np.pi, 10)]
    points = analyzer.find_critical_points(guesses)

    # Sort by q-coordinate for consistent plotting
    points_sorted = sorted(points, key=lambda x: x[0])

    for i in range(len(trajectories)):
        if i < len(points_sorted):
            # Store (q, p, alpha, type)
            cp_data = analyzer.analyze_point(points_sorted[i])
            trajectories[i].append((points_sorted[i][0], points_sorted[i][1], alpha, cp_data.singularity_type))
        else:
            trajectories[i].append((None, None, alpha, None))

# --- Animation ---
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(-np.pi, np.pi)
ax.set_ylim(-2.5, 2.5) # Adjusted for potential
ax.set_xlabel('Position $q$', fontsize=12)
ax.set_ylabel('Energy / Potential $V(q)$', fontsize=12)
ax.set_title(r'Symplectic Dance: Bifurcations of $H(q,p) = \frac{p^2}{2} + \cos q + \alpha \cos 2q$', fontsize=14)
ax.grid(True, alpha=0.3)

# Graphical elements
line_v, = ax.plot([], [], 'k-', lw=1.5, alpha=0.6, label='Potential $V(q)$')
dots = [ax.plot([], [], 'o', markersize=10, markeredgecolor='black')[0] for i in range(len(trajectories))]
alpha_text = ax.text(0.05, 0.92, '', transform=ax.transAxes, fontsize=12,
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

def init():
    line_v.set_data([], [])
    for dot in dots:
        dot.set_data([], [])
    alpha_text.set_text('')
    return [line_v, alpha_text] + dots

def animate(frame):
    alpha = alpha_vals[frame]

    # Update the potential in the background
    q_grid = np.linspace(-np.pi, np.pi, 200)
    v_grid = np.cos(q_grid) + alpha * np.cos(2*q_grid)
    line_v.set_data(q_grid, v_grid)

    alpha_text.set_text(f'$\\alpha = {alpha:.2f}$')

    for i, dot in enumerate(dots):
        q_val, p_val, a_val, s_type = trajectories[i][frame]
        if q_val is not None:
            # Place the point on the potential curve for visibility
            v_val = np.cos(q_val) + alpha * np.cos(2*q_val)
            dot.set_data([q_val], [v_val])

            # Color change based on stability (Hessian)
            # A minimum is a stable center, a maximum/saddle is unstable
            dot.set_color('red' if i % 2 == 0 else 'blue')
        else:
            dot.set_data([], [])

    return [line_v, alpha_text] + dots

anim = FuncAnimation(fig, animate, init_func=init, frames=len(alpha_vals),
                    interval=80, blit=True)

# --- Bifurcation Diagram ---
plt.figure(figsize=(10, 6))
for i in range(len(trajectories)):
    q_coords = [t[0] for t in trajectories[i] if t[0] is not None]
    a_coords = [t[2] for t in trajectories[i] if t[0] is not None]
    if q_coords:
        plt.scatter(a_coords, q_coords, s=10, label=f'Branch {i+1}')

plt.xlabel(r'Bifurcation parameter $\alpha$', fontsize=13)
plt.ylabel('Fixed points position $q$', fontsize=13)
plt.title('Bifurcation Diagram: Creation/Annihilation of Critical Points', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()

from IPython.display import HTML
HTML(anim.to_jshtml())



## Geometric Plating of Contributions

In [ ]:
# 2d_geometric_plating.py
import matplotlib.colors as mcolors

x, y = sp.symbols('x y')

# Rotationally perturbed harmonic oscillator
beta = 0.4  # Cubic perturbation strength
phi = (x**2 + y**2)/2 + beta * (x**3 - 3*x*y**2)  # Real part of (x+iy)^3
amp = 1.0

analyzer = Analyzer(phi, amp, [x, y], domain=[(-2, 2), (-2, 2)])
points = analyzer.find_critical_points([
    np.array([g, h]) for g in np.linspace(-1.5, 1.5, 5) 
                     for h in np.linspace(-1.5, 1.5, 5)
])

print("Critical points and their contributions at λ=100:")
evaluator = AsymptoticEvaluator()
lam_test = 100

contributions = []
for p in points:
    cp = analyzer.analyze_point(p)
    res = evaluator.evaluate(cp, lam_test)
    contributions.append((p, cp, res.total_value))
    print(f"  ({p[0]:.3f}, {p[1]:.3f}) | Type: {cp.singularity_type.value:8s} | "
          f"|I| = {np.abs(res.total_value):.4f} | arg = {np.angle(res.total_value):.2f} rad")

# Create "plating" visualization: each contribution as a colored disk
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Left: Phase landscape with critical points
x_bg = np.linspace(-1.8, 1.8, 150)
y_bg = np.linspace(-1.8, 1.8, 150)
X_bg, Y_bg = np.meshgrid(x_bg, y_bg)
Z_bg = analyzer.func_phase(X_bg, Y_bg)

ax[0].contourf(X_bg, Y_bg, Z_bg, levels=40, cmap='viridis', alpha=0.9)
for p, cp, val in contributions:
    size = 800 * np.abs(val) / max(np.abs(v) for _,_,v in contributions)
    hue = (np.angle(val) + np.pi) / (2*np.pi)  # Map phase to [0,1]
    color = plt.cm.hsv(hue)
    ax[0].scatter(p[0], p[1], s=size, c=[color], 
                 edgecolors='white', linewidths=1.5, zorder=10)
ax[0].set_xlabel('x', fontsize=12)
ax[0].set_ylabel('y', fontsize=12)
ax[0].set_title('Phase Landscape with Contribution "Ingredients"', fontsize=13)
ax[0].axis('equal')
ax[0].grid(True, alpha=0.3)

# Right: Geometric plating arrangement
ax[1].set_xlim(-1.5, 1.5)
ax[1].set_ylim(-1.5, 1.5)
ax[1].set_aspect('equal')
ax[1].axis('off')
ax[1].set_title('Geometric Plating: Superposition of Asymptotic Contributions', 
               fontsize=14, pad=20)

# Arrange contributions in a circular "plate"
n = len(contributions)
for i, (p, cp, val) in enumerate(contributions):
    angle = 2 * np.pi * i / n
    radius = 0.9
    x_pos = radius * np.cos(angle)
    y_pos = radius * np.sin(angle)
    
    # Disk size proportional to magnitude
    size = 1200 * np.abs(val) / max(np.abs(v) for _,_,v in contributions)
    
    # Color by phase (argument)
    hue = (np.angle(val) + np.pi) / (2*np.pi)
    saturation = 0.9
    value = 0.95
    color = mcolors.hsv_to_rgb((hue, saturation, value))
    
    # Draw contribution disk
    circle = plt.Circle((x_pos, y_pos), np.sqrt(size)/60, 
                       color=color, ec='white', linewidth=2)
    ax[1].add_patch(circle)
    
    # Label with magnitude
    ax[1].text(x_pos, y_pos, f'{np.abs(val):.2f}', 
              ha='center', va='center', fontsize=9, 
              color='white' if np.abs(val) > 0.03 else 'black',
              fontweight='bold')
    
    # Connect to center with delicate line
    ax[1].plot([0, x_pos], [0, y_pos], 'w-', alpha=0.3, linewidth=1)

# Center: total contribution
total_val = sum(val for _,_,val in contributions)
total_size = 1500 * np.abs(total_val) / max(np.abs(v) for _,_,v in contributions)
hue_total = (np.angle(total_val) + np.pi) / (2*np.pi)
color_total = mcolors.hsv_to_rgb((hue_total, 0.95, 1.0))
circle_total = plt.Circle((0, 0), np.sqrt(total_size)/60, 
                         color=color_total, ec='gold', linewidth=3)
ax[1].add_patch(circle_total)
ax[1].text(0, 0, f'Total\n{np.abs(total_val):.2f}', 
          ha='center', va='center', fontsize=11, 
          color='white', fontweight='bold')

# Add plate boundary
plate = plt.Circle((0, 0), 1.2, fill=False, ec='gray', linewidth=4, linestyle='--')
ax[1].add_patch(plate)

plt.tight_layout()
plt.show()

# Show convergence of total vs λ
lambdas = np.logspace(1, 3, 30)
total_mags = []

for lam in lambdas:
    total = 0j
    for p, cp, _ in contributions:
        res = evaluator.evaluate(cp, lam)
        total += res.total_value
    total_mags.append(np.abs(total))

plt.figure(figsize=(10, 5))
plt.loglog(lambdas, total_mags, 'o-', linewidth=2.5, markersize=6)
plt.xlabel(r'$\lambda$', fontsize=12)
plt.ylabel(r'Total magnitude $|\sum I_k(\lambda)|$', fontsize=12)
plt.title('Convergence of Superposed Asymptotic Contributions', fontsize=14)
plt.grid(True, which="both", ls=":", alpha=0.7)
plt.tight_layout()
plt.show()

## Ritual of Convergence

In [ ]:
# 2d_ritual_convergence.py
import matplotlib.patches as mpatches


x, y = sp.symbols('x y')

# Mixed singularity: Airy in x, Morse in y → total scaling λ^{-5/6}
phi = x**3/3 + y**2/2
amp = 1.0

analyzer = Analyzer(phi, amp, [x, y], domain=[(-2, 2), (-2, 2)])
points = analyzer.find_critical_points([np.array([0.0, 0.0])])

if not points:
    raise ValueError("No critical point found!")
cp = analyzer.analyze_point(points[0])
print(f"Critical point: ({cp.position[0]:.3f}, {cp.position[1]:.3f})")
print(f"Singularity type: {cp.singularity_type.value}")
print(f"Canonical coefficients: {cp.canonical_coefficients}")

# Ritual convergence plot
evaluator = AsymptoticEvaluator()
lambdas = np.logspace(1, 3.5, 60)

magnitudes = []
for lam in lambdas:
    res = evaluator.evaluate(cp, lam)
    magnitudes.append(np.abs(res.leading_term))

# Create ritual visualization
fig, ax = plt.subplots(figsize=(12, 8))

# Main convergence curve
ax.loglog(lambdas, magnitudes, 'o-', color='#2c3e50', 
         linewidth=3, markersize=8, alpha=0.9, label='Asymptotic magnitude')

# Theoretical slope lines
lambda_ref = 30
mag_ref = np.abs(evaluator.evaluate(cp, lambda_ref).leading_term)
slope = -5/6  # Airy 2D scaling

lambda_line = np.array([lambda_ref, lambdas[-1]])
mag_line = mag_ref * (lambda_line / lambda_ref)**slope
ax.loglog(lambda_line, mag_line, 'r--', linewidth=2.5, 
         label=f'Theoretical slope = {slope:.3f}')

# Ritual zones with symbolic meaning
zone1 = plt.Polygon([[10, 1e-1], [30, 1e-1], [30, 1e-3], [10, 1e-3]], 
                   color='gold', alpha=0.2)
zone2 = plt.Polygon([[30, 1e-1], [100, 1e-1], [100, 1e-4], [30, 1e-4]], 
                   color='green', alpha=0.2)
zone3 = plt.Polygon([[100, 1e-1], [3000, 1e-1], [3000, 1e-6], [100, 1e-6]], 
                   color='blue', alpha=0.2)

ax.add_patch(zone1)
ax.add_patch(zone2)
ax.add_patch(zone3)

ax.text(18, 3e-2, 'I. Veil of\nOscillation', fontsize=11, 
       fontweight='bold', ha='center', color='#8b4513')
ax.text(60, 3e-3, 'II. Threshold\nof Geometry', fontsize=11, 
       fontweight='bold', ha='center', color='darkgreen')
ax.text(400, 3e-5, 'III. Realm of\nAsymptotics', fontsize=11, 
       fontweight='bold', ha='center', color='darkblue')

# Decorative elements
ax.axvline(30, color='orange', ls='--', alpha=0.6, linewidth=2)
ax.axvline(100, color='green', ls='--', alpha=0.6, linewidth=2)

ax.set_xlabel(r'Ritual parameter $\lambda$ (symbolic time)', fontsize=14)
ax.set_ylabel(r'Sacred magnitude $|I(\lambda)|$', fontsize=14)
ax.set_title('Ritual of Convergence: Emergence of Geometry from Oscillation\n'
            r'Critical point type: Airy$_{2D}$  $\rightarrow$  Scaling: $\lambda^{-5/6}$',
            fontsize=15, pad=20)
ax.grid(True, which="both", ls=":", alpha=0.5)
ax.legend(loc='lower left', fontsize=12)

# Add symbolic markers at key λ values
for lam_mark in [10, 30, 100, 300, 1000]:
    mag_mark = np.abs(evaluator.evaluate(cp, lam_mark).leading_term)
    ax.plot(lam_mark, mag_mark, 'o', color='purple', markersize=10, zorder=10)
    ax.text(lam_mark, mag_mark*1.8, f'λ={lam_mark}', 
           ha='center', fontsize=9, rotation=0)

plt.tight_layout()
plt.show()

# Phase evolution ritual
phases = []
for lam in lambdas:
    res = evaluator.evaluate(cp, lam)
    phases.append(np.angle(res.leading_term))

plt.figure(figsize=(12, 5))
plt.semilogx(lambdas, phases, 'o-', color='#8e44ad', linewidth=2.5, markersize=6)
plt.axhline(np.pi/6 + np.pi/4, color='r', ls='--', alpha=0.7, 
           label='Theoretical phase: π/6 (Airy) + π/4 (Morse)')
plt.xlabel(r'$\lambda$', fontsize=12)
plt.ylabel(r'Sacred phase $\arg I(\lambda)$', fontsize=12)
plt.title('Phase Ritual: Stabilization of Maslov Index', fontsize=14)
plt.grid(True, which="both", ls=":", alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

## Dream of an Integral (REM cycles animation)

In [ ]:
# 2d_dream_integral.py

x, y = sp.symbols('x y')

# Time-dependent phase with bifurcation parameter t
t_vals = np.linspace(-2.0, 2.0, 80)  # Dream timeline

# Storage for dream narrative
dream_data = []

for t in t_vals:
    # Phase with pitchfork bifurcation: φ = x⁴/4 + t·x²/2 + y²/2
    phi = x**4/4 + t * x**2/2 + y**2/2
    amp = 1.0
    
    analyzer = Analyzer(phi, amp, [x, y], domain=[(-2, 2), (-2, 2)])
    guesses = [np.array([g, 0.0]) for g in np.linspace(-1.8, 1.8, 7)]
    points = analyzer.find_critical_points(guesses)
    
    # Classify points
    morses = []
    degenerates = []
    for p in points:
        cp = analyzer.analyze_point(p)
        if cp.singularity_type == 'morse':
            morses.append((p, cp))
        else:
            degenerates.append((p, cp))
    
    dream_data.append({
        't': t,
        'morses': morses,
        'degenerates': degenerates,
        'analyzer': analyzer
    })

# Create dream animation
fig = plt.figure(figsize=(14, 8))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

ax_phase = fig.add_subplot(gs[:, :2])  # Large phase plot
ax_mag = fig.add_subplot(gs[0, 2])     # Magnitude timeline
ax_narr = fig.add_subplot(gs[1, 2])    # Narrative text

# Setup phase plot
ax_phase.set_xlim(-2, 2)
ax_phase.set_ylim(-2, 2)
ax_phase.set_xlabel('x', fontsize=12)
ax_phase.set_ylabel('y', fontsize=12)
ax_phase.set_title('Dreamscape: Evolution of Critical Points', fontsize=14)
ax_phase.grid(True, alpha=0.3)

# Setup magnitude plot
mag_timeline = []
t_timeline = []
line_mag, = ax_mag.semilogy([], [], 'b-', linewidth=2.5)
ax_mag.set_xlim(-2.1, 2.1)
ax_mag.set_ylim(1e-3, 10)
ax_mag.set_xlabel('Dream time t', fontsize=10)
ax_mag.set_ylabel('|I(λ=100)|', fontsize=10)
ax_mag.set_title('Integral Magnitude', fontsize=11)
ax_mag.grid(True, which="both", ls=":", alpha=0.5)

# Setup narrative
ax_narr.axis('off')
narr_text = ax_narr.text(0.5, 0.5, '', ha='center', va='center', 
                        fontsize=11, wrap=True,
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9))

# Precompute magnitude timeline
evaluator = AsymptoticEvaluator()
lam_dream = 100
for data in dream_data:
    total = 0j
    for p, cp in data['morses'] + data['degenerates']:
        res = evaluator.evaluate(cp, lam_dream)
        total += res.total_value
    mag_timeline.append(np.abs(total))
    t_timeline.append(data['t'])

def init():
    return []

def animate(frame):
    data = dream_data[frame]
    t = data['t']
    
    # Clear phase plot
    ax_phase.clear()
    ax_phase.set_xlim(-2, 2)
    ax_phase.set_ylim(-2, 2)
    ax_phase.set_xlabel('x', fontsize=12)
    ax_phase.set_ylabel('y', fontsize=12)
    ax_phase.grid(True, alpha=0.3)
    
    # Plot phase background
    x_bg = np.linspace(-2, 2, 100)
    y_bg = np.linspace(-2, 2, 100)
    X_bg, Y_bg = np.meshgrid(x_bg, y_bg)
    Z_bg = data['analyzer'].func_phase(X_bg, Y_bg)
    ax_phase.contourf(X_bg, Y_bg, Z_bg, levels=30, cmap='plasma', alpha=0.7)
    
    # Plot critical points with dream symbolism
    for p, cp in data['morses']:
        if cp.hessian_matrix[0,0] > 0:  # Minimum
            ax_phase.plot(p[0], p[1], 'go', markersize=14, 
                         label='Stable dream element' if frame==0 else "")
        else:  # Saddle
            ax_phase.plot(p[0], p[1], 'rs', markersize=14, 
                         label='Unstable transition' if frame==0 else "")
    
    for p, cp in data['degenerates']:
        ax_phase.plot(p[0], p[1], '*', color='gold', markersize=20,
                     label='Dream singularity' if frame==0 else "")
    
    # Dream narrative based on t
    if t < -1.2:
        narrative = "LATENT STATE\nDream elements dormant\nin potential landscape"
        bg_color = '#1a237e'
    elif t < -0.3:
        narrative = "DREAM ONSET\nCritical points emerge\nfrom symmetry breaking"
        bg_color = '#283593'
    elif t < 0.3:
        narrative = "SINGULAR MOMENT\nFusion into catastrophe\nthreshold of awareness"
        bg_color = '#8e24aa'
    elif t < 1.2:
        narrative = "DREAM UNFOLDING\nNew structures crystallize\nin conscious space"
        bg_color = '#d81b60'
    else:
        narrative = "INTEGRATION\nStable dream architecture\nresolves into memory"
        bg_color = '#5e35b1'
    
    ax_phase.set_title(f'Dream Time t = {t:.2f}  |  φ = x⁴/4 + ({t:.2f})·x²/2 + y²/2', 
                      fontsize=13, pad=15)
    ax_phase.legend(loc='upper right', fontsize=9)
    
    # Update magnitude plot
    line_mag.set_data(t_timeline[:frame+1], mag_timeline[:frame+1])
    
    # Update narrative
    ax_narr.clear()
    ax_narr.axis('off')
    ax_narr.set_xlim(0, 1)
    ax_narr.set_ylim(0, 1)
    ax_narr.add_patch(plt.Rectangle((0, 0), 1, 1, color=bg_color, alpha=0.2))
    ax_narr.text(0.5, 0.5, narrative, ha='center', va='center', 
                fontsize=12, fontweight='bold', wrap=True)
    
    return []

anim = FuncAnimation(fig, animate, init_func=init, frames=len(t_vals),
                    interval=100, blit=False)

from IPython.display import HTML
HTML(anim.to_jshtml())


## Fresnel Diffraction at a Straight Edge

In [ ]:
# fresnel_diffraction_edge.py
from scipy.special import fresnel

# Coordinate along the observation screen (normalized)
x = sp.symbols('x', real=True)

# Fresnel integral for straight edge diffraction:
# I(X) = ∫_{-∞}^{∞} Θ(t) exp(iπ(t - X)²/2) dt
# where Θ is Heaviside step function → integral from 0 to ∞
# Phase function after change of variables: φ(t) = (t - X)²/2
# Critical point: t_c = X (only contributes if X > 0, i.e., in illuminated region)

X_param = 0.8  # Position relative to edge (X=0)

# Phase centered at observation point X
phi = (x - X_param)**2 / 2
amp = 1.0  # Constant amplitude (Heaviside handled by domain)

# Morse point at x = X_param
analyzer = Analyzer(phi, amp, [x], domain=[(-5, 5)])
points = analyzer.find_critical_points([np.array([X_param])])

print(f"Fresnel diffraction at X = {X_param}")
print(f"Critical point: x_c = {points[0][0]:.4f}")
cp = analyzer.analyze_point(points[0])
print(f"Type: {cp.singularity_type.value} | Hessian: {cp.hessian_matrix[0,0]:.4f}")

# Exact Fresnel integral solution:
# I(X) = (1+i)/2 [C(X√(2/π)) + i S(X√(2/π))] + 1/2
def exact_fresnel_edge(X):
    s, c = fresnel(X * np.sqrt(2/np.pi))
    return 0.5 + 0.5*(1+1j)*(c + 1j*s)

# Asymptotic evaluation for large frequency parameter λ
# Physical scaling: λ = 2π/λ_wavelength in normalized units
evaluator = AsymptoticEvaluator()
lambdas = np.logspace(0.5, 2.5, 30)  # λ from ~3 to ~300

asymptotic_vals = []
exact_vals = []

for lam in lambdas:
    # Stationary phase integral: ∫ exp(i λ φ(x)) dx
    cp = analyzer.analyze_point(points[0])
    res = evaluator.evaluate(cp, lam)
    asymptotic_vals.append(res.total_value)
    
    # Exact solution scaled by √(2π/λ) for comparison
    exact_vals.append(exact_fresnel_edge(X_param) * np.sqrt(2*np.pi/lam) * 
                     np.exp(1j * lam * cp.phase_value) * np.exp(1j*np.pi/4))

# Plot comparison
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.loglog(lambdas, np.abs(asymptotic_vals), 'o-', label='Asymptotic', linewidth=2)
plt.loglog(lambdas, np.abs(exact_vals), 's--', label='Exact (scaled)', linewidth=2)
plt.xlabel(r'Frequency parameter $\lambda$', fontsize=12)
plt.ylabel(r'Magnitude $|I(\lambda)|$', fontsize=12)
plt.title('Fresnel Edge Diffraction: Magnitude', fontsize=13)
plt.grid(True, which="both", ls=":", alpha=0.7)
plt.legend()

plt.subplot(1, 2, 2)
plt.semilogx(lambdas, np.angle(asymptotic_vals), 'o-', label='Asymptotic', linewidth=2)
plt.semilogx(lambdas, np.angle(exact_vals), 's--', label='Exact (scaled)', linewidth=2)
plt.xlabel(r'Frequency parameter $\lambda$', fontsize=12)
plt.ylabel(r'Phase $\arg I(\lambda)$', fontsize=12)
plt.title('Phase Comparison', fontsize=13)
plt.grid(True, which="both", ls=":", alpha=0.7)
plt.legend()

plt.suptitle(f'Straight Edge Diffraction at X = {X_param}\n'
             r'$I(X) = \int_0^\infty e^{i\lambda (t-X)^2/2} dt$', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Physical interpretation plot: intensity vs position X at fixed λ
X_vals = np.linspace(-2, 2, 200)
lam_fixed = 50
intensity_asym = []
intensity_exact = []

for X in X_vals:
    phi_X = (x - X)**2 / 2
    analyzer_X = Analyzer(phi_X, 1.0, [x], domain=[(-5, 5)])
    pts = analyzer_X.find_critical_points([np.array([X])])
    
    if pts and pts[0][0] > 0:  # Critical point in integration domain (X > 0)
        cp_X = analyzer_X.analyze_point(pts[0])
        res = evaluator.evaluate(cp_X, lam_fixed)
        intensity_asym.append(np.abs(res.total_value)**2)
    else:  # No stationary point in domain → rapid oscillations → near zero
        intensity_asym.append(0.0)
    
    # Exact intensity
    exact_val = exact_fresnel_edge(X)
    intensity_exact.append(np.abs(exact_val)**2)

plt.figure(figsize=(10, 6))
plt.plot(X_vals, intensity_asym, 'r-', linewidth=2.5, label=f'Asymptotic (λ={lam_fixed})')
plt.plot(X_vals, intensity_exact, 'b--', linewidth=2, label='Exact Fresnel integral')
plt.axvline(0, color='k', ls=':', alpha=0.7, label='Edge position (X=0)')
plt.fill_between(X_vals, 0, max(intensity_exact), where=(X_vals<0), 
                 color='gray', alpha=0.2, label='Geometric shadow')
plt.xlabel('Position X (normalized)', fontsize=12)
plt.ylabel('Normalized intensity', fontsize=12)
plt.title('Fresnel Diffraction Pattern: Shadow → Illuminated Transition', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Fourier Transform of a Perturbed Gaussian

In [ ]:
# fourier_perturbed_gaussian.py

x = sp.symbols('x', real=True)

# Signal: f(x) = exp(-x²/2 - ε x⁴/4)
# Fourier transform: F(k) = ∫ exp(-x²/2 - ε x⁴/4) exp(-i k x) dx
# Phase for stationary phase: φ(x) = -i(-x²/2 - ε x⁴/4) - i k x → actually we treat as
# Complex phase: but for asymptotic analysis of FT at large k, we use φ(x) = kx (oscillatory part)
# and amplitude a(x) = exp(-x²/2 - ε x⁴/4)

k_param = 3.0  # Fourier frequency (large k → stationary phase regime)
epsilon = 0.2  # Anharmonicity strength

# For FT at frequency k: integral ∫ a(x) exp(-i k x) dx
# Phase function for stationary phase method: φ(x) = -x  (since exp(-i k x) = exp(i λ φ) with λ=k, φ=-x)
# But critical point condition: dφ/dx = 0 → no solution! 
# Correction: we must include the amplitude's phase contribution via steepest descent
# Instead, treat as complex integral and find saddle points of S(x) = -x²/2 - εx⁴/4 - i k x

# Complex saddle point equation: dS/dx = -x - ε x³ - i k = 0
# For small ε, solution: x_c ≈ -i k (harmonic) + correction

# Construct effective phase for stationary phase in complex plane:
# We rotate contour to pass through saddle point → real phase function along steepest descent
# Approximate saddle: x_c ≈ -i k + i ε k³ (from perturbation theory)

# For demonstration, we use real-valued proxy that captures the physics:
# Consider φ(x) = x²/2 + ε x⁴/4 + k x  (real phase whose critical point mimics saddle location)
phi_total = -x**2/2 - epsilon * x**4/4 +  k_param * x
amp = 1.0  # Amplitude absorbed into phase approximation

analyzer = Analyzer(phi, amp, [x], domain=[(-5, 5)])
points = analyzer.find_critical_points([np.array([-k_param])])  # Initial guess near -k

print(f"Fourier Transform at k = {k_param} with anharmonicity ε = {epsilon}")
if points:
    cp = analyzer.analyze_point(points[0])
    print(f"Critical point: x_c = {cp.position[0]:.4f}")
    print(f"Type: {cp.singularity_type.value}")
    print(f"Phase value: φ(x_c) = {cp.phase_value:.4f}")
    
    # Evaluate asymptotic contribution
    evaluator = AsymptoticEvaluator()
    lam = 1.0  # Here λ=1 since k is already in phase
    res = evaluator.evaluate(cp, lam)
    
    print(f"\nAsymptotic FT magnitude: |F(k)| ≈ {np.abs(res.total_value):.4f}")
    print(f"Phase: arg F(k) ≈ {np.angle(res.total_value):.4f} rad")
else:
    print("No critical point found (try different initial guess)")

# Compare with numerical Fourier transform for validation
def signal(x, eps):
    return np.exp(-x**2/2 - eps * x**4/4)

def numerical_ft(k, eps, x_max=10, n_pts=20000):
    x = np.linspace(-x_max, x_max, n_pts)
    dx = x[1] - x[0]
    fx = signal(x, eps)
    integrand = fx * np.exp(-1j * k * x)
    return np.trapezoid(integrand, x)

k_vals = np.linspace(0, 6, 60)
ft_numerical = [numerical_ft(k, epsilon) for k in k_vals]
ft_magnitude = np.abs(ft_numerical)

# Asymptotic prediction using stationary phase at each k
ft_asymptotic = []
for k in k_vals:
    phi_k = x**2/2 + epsilon * x**4/4 + k * x
    analyzer_k = Analyzer(phi_k, 1.0, [x], domain=[(-10, 10)])
    pts = analyzer_k.find_critical_points([np.array([-k])])
    if pts:
        cp_k = analyzer_k.analyze_point(pts[0])
        res_k = evaluator.evaluate(cp_k, 1.0)
        ft_asymptotic.append(np.abs(res_k.total_value))
    else:
        ft_asymptotic.append(0.0)

plt.figure(figsize=(10, 6))
plt.plot(k_vals, ft_magnitude, 'b-', linewidth=2.5, label='Numerical FT')
plt.plot(k_vals, ft_asymptotic, 'ro', markersize=6, alpha=0.7, label='Stationary phase asymptotic')
plt.xlabel('Frequency $k$', fontsize=12)
plt.ylabel(r'Magnitude $|\mathcal{F}(k)|$', fontsize=12)
plt.title(f'Fourier Transform of Anharmonic Gaussian\n'
          r'$f(x) = \exp(-x^2/2 - \epsilon x^4/4)$,  $\epsilon = {epsilon}$', 
          fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Show how anharmonicity shifts the critical point from harmonic prediction
k_test = np.linspace(0.5, 5, 20)
x_c_harmonic = [-k for k in k_test]  # Harmonic oscillator: x_c = -k
x_c_anharmonic = []

for k in k_test:
    phi_k = x**2/2 + epsilon * x**4/4 + k * x
    analyzer_k = Analyzer(phi_k, 1.0, [x], domain=[(-10, 10)])
    pts = analyzer_k.find_critical_points([np.array([-k])])
    if pts:
        x_c_anharmonic.append(pts[0][0])
    else:
        x_c_anharmonic.append(np.nan)

plt.figure(figsize=(10, 5))
plt.plot(k_test, x_c_harmonic, 'b--', linewidth=2, label='Harmonic prediction: $x_c = -k$')
plt.plot(k_test, x_c_anharmonic, 'ro-', linewidth=2, markersize=6, label='Anharmonic critical point')
plt.xlabel('Frequency parameter $k$', fontsize=12)
plt.ylabel('Critical point position $x_c$', fontsize=12)
plt.title('Critical Point Shift Due to Anharmonicity', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Asymptotics of Bessel Functions

In [ ]:
"""
bessel_debye_asymptotic.py

Debye asymptotic expansion of J_ν(νz) using stationary phase.

We use the exact Poisson integral representation:

    J_ν(νz) = (1 / 2π) ∫_{-π}^{π} exp(i ν φ(t)) dt

with phase
    φ(t) = z sin(t) − t

For z > 1, two real stationary points exist and produce
the classical Debye cosine asymptotic.
"""

from scipy.special import jv


# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------

z_param = 3        # Must be > 1 for oscillatory regime
nu_test = 50         # Single test value
nu_values = np.linspace(10, 200, 40)

t = sp.symbols('t', real=True)

# Correct Poisson phase
phi = z_param * sp.sin(t) - t
amp = 1.0

# ------------------------------------------------------------
# Stationary phase analysis (geometry independent of ν)
# ------------------------------------------------------------

analyzer = Analyzer(phi, amp, [t], domain=[(-np.pi, np.pi)])

# Stationary points satisfy:
# φ'(t) = z cos(t) - 1 = 0
# → cos(t_c) = 1/z
t_c = np.arccos(1 / z_param)

points = analyzer.find_critical_points(
    [np.array([t_c]), np.array([-t_c])]
)

critical_points = [analyzer.analyze_point(p) for p in points]

print("\nDebye Asymptotics via Stationary Phase")
print(f"z = {z_param}")
print(f"Stationary points found: {len(critical_points)}")

for cp in critical_points:
    print(f"\nt_c = {cp.position[0]:.6f}")
    print(f"  Type: {cp.singularity_type.value}")
    print(f"  Hessian: {cp.hessian_matrix[0,0]:.6f}")
    print(f"  Phase value φ(t_c): {cp.phase_value.real:.6f}")


# ------------------------------------------------------------
# Single ν test
# ------------------------------------------------------------

evaluator = AsymptoticEvaluator()

total_contribution = 0j
for cp in critical_points:
    res = evaluator.evaluate(cp, nu_test)
    total_contribution += res.total_value

# Multiply by exact Poisson prefactor
asymptotic_value = (1 / (2*np.pi)) * np.real(total_contribution)

exact_value = jv(nu_test, nu_test * z_param)

print("\n--- Single ν Test ---")
print(f"Exact J_{nu_test}({nu_test*z_param}) = {exact_value:.6e}")
print(f"Asymptotic approximation            = {asymptotic_value:.6e}")
print(f"Relative error                     = "
      f"{abs(asymptotic_value - exact_value)/abs(exact_value):.2%}")


# ------------------------------------------------------------
# Convergence study
# ------------------------------------------------------------

exact_vals = []
asym_vals = []

for nu in nu_values:

    total = 0j
    for cp in critical_points:
        res = evaluator.evaluate(cp, nu)
        total += res.total_value

    asym_val = (1 / (2*np.pi)) * np.real(total)

    exact_vals.append(jv(nu, nu*z_param))
    asym_vals.append(asym_val)


# ------------------------------------------------------------
# Plot comparison
# ------------------------------------------------------------

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(nu_values, exact_vals, 'b-', linewidth=2.5, label='Exact (scipy)')
plt.plot(nu_values, asym_vals, 'ro-', markersize=4,
         label='Stationary phase asymptotic')
plt.xlabel('Order ν')
plt.ylabel(r'$J_\nu(\nu z)$')
plt.title(f'Debye Asymptotics (z={z_param})')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1,2,2)
rel_error = np.abs((np.array(asym_vals) - np.array(exact_vals))
                   / np.array(exact_vals))
plt.loglog(nu_values, rel_error, 'm^-', linewidth=2)
plt.xlabel('Order ν')
plt.ylabel('Relative error')
plt.title('Asymptotic Convergence')
plt.grid(True, which="both", ls=":")

plt.tight_layout()
plt.show()


## Debye asymptotics

In [ ]:
"""
bessel_debye_asymptotic_order2.py

Debye asymptotics of J_ν(νz) using stationary phase.

Compares:
    - Leading Morse term (O(ν^{-1}))
    - Second-order corrected Morse expansion (O(ν^{-2}))
"""


# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------

z_param = 5
t = sp.symbols('t', real=True)

# Use correct Poisson phase
phi = z_param * sp.sin(t) - t
amp = 1.0

# High-resolution ν sampling
nu_values = np.linspace(20, 400, 150)

# ------------------------------------------------------------
# Stationary phase geometry (independent of ν)
# ------------------------------------------------------------

analyzer = Analyzer(phi, amp, [t], domain=[(-np.pi, np.pi)])

t_c = np.arccos(1 / z_param)
points = analyzer.find_critical_points(
    [np.array([t_c]), np.array([-t_c])]
)

critical_points = [analyzer.analyze_point(p) for p in points]

evaluator = AsymptoticEvaluator()

# ------------------------------------------------------------
# Compute asymptotics
# ------------------------------------------------------------

exact_vals = []
leading_vals = []
corrected_vals = []

for nu in nu_values:

    total_leading = 0j
    total_corrected = 0j

    for cp in critical_points:
        res = evaluator.evaluate(cp, nu)

        total_leading += res.leading_term
        total_corrected += res.total_value   # includes order-2 correction

    # Multiply by Poisson prefactor
    leading_vals.append((1/(2*np.pi)) * np.real(total_leading))
    corrected_vals.append((1/(2*np.pi)) * np.real(total_corrected))
    exact_vals.append(jv(nu, nu*z_param))

exact_vals = np.array(exact_vals)
leading_vals = np.array(leading_vals)
corrected_vals = np.array(corrected_vals)

# ------------------------------------------------------------
# Plot comparison
# ------------------------------------------------------------

plt.figure(figsize=(13,5))

plt.subplot(1,2,1)
plt.plot(nu_values, exact_vals, 'k-', linewidth=2.5, label='Exact (scipy)')
plt.plot(nu_values, leading_vals, 'r--', linewidth=1.8,
         label='Leading Morse (O(ν⁻¹))')
plt.plot(nu_values, corrected_vals, 'b-', linewidth=2,
         label='Second-order Morse (O(ν⁻²))')

plt.xlabel('Order ν')
plt.ylabel(r'$J_\nu(\nu z)$')
plt.title(f'Debye Asymptotics with Order Comparison (z={z_param})')
plt.grid(True, alpha=0.3)
plt.legend()


plt.subplot(1,2,2)

rel_error_leading = np.abs((leading_vals - exact_vals)/exact_vals)
rel_error_corrected = np.abs((corrected_vals - exact_vals)/exact_vals)

plt.loglog(nu_values, rel_error_leading, 'r--', linewidth=2,
           label='Leading error')
plt.loglog(nu_values, rel_error_corrected, 'b-', linewidth=2,
           label='Corrected error')

# Reference slopes
ref = 1/nu_values
plt.loglog(nu_values, ref, 'k:', alpha=0.6, label='O(1/ν)')
plt.loglog(nu_values, ref**2, 'k-.', alpha=0.6, label='O(1/ν²)')

plt.xlabel('Order ν')
plt.ylabel('Relative error')
plt.title('Asymptotic Convergence Rates')
plt.grid(True, which="both", ls=":")
plt.legend()

plt.tight_layout()
plt.show()


## 2D stationary phase 

In [ ]:
"""
2D stationary phase example.

Demonstrates:
    - 2D Morse critical point
    - Leading term O(λ^{-1})
    - Second-order correction O(λ^{-2})
    - Empirical slope verification
"""


# ------------------------------------------------------------
# Define 2D phase
# ------------------------------------------------------------

x, y = sp.symbols('x y', real=True)

epsilon = 0.2

phi = x**2/2 + y**2/2 + epsilon * x**3
amp = 1.0

analyzer = Analyzer(phi, amp, [x, y])
points = analyzer.find_critical_points([np.array([0.0, 0.0])])
cp = analyzer.analyze_point(points[0])

print("\n2D Stationary Phase Example")
print(f"Critical point: {cp.position}")
print(f"Type: {cp.singularity_type.value}")
print(f"Hessian determinant: {cp.hessian_det:.4f}")
print(f"Signature: {cp.signature}")


# ------------------------------------------------------------
# Evaluate asymptotics
# ------------------------------------------------------------

evaluator = AsymptoticEvaluator()

lambda_vals = np.logspace(1, 3, 120)

leading_vals = []
corrected_vals = []

for lam in lambda_vals:

    res = evaluator.evaluate(cp, lam)

    leading_vals.append(abs(res.leading_term))
    corrected_vals.append(abs(res.total_value))

leading_vals = np.array(leading_vals)
corrected_vals = np.array(corrected_vals)


# ------------------------------------------------------------
# Plot decay rates
# ------------------------------------------------------------

plt.figure(figsize=(10,6))

plt.loglog(lambda_vals, leading_vals, 'r--', linewidth=2,
           label='Leading term |I₀(λ)|')

plt.loglog(lambda_vals, corrected_vals, 'b-', linewidth=2,
           label='Corrected term |I₀+I₁|')

# Reference slopes
plt.loglog(lambda_vals, 1/lambda_vals,
           'k:', alpha=0.6, label='O(λ⁻¹)')
plt.loglog(lambda_vals, 1/lambda_vals**2,
           'k-.', alpha=0.6, label='O(λ⁻²)')

plt.xlabel('λ')
plt.ylabel('Magnitude')
plt.title('2D Stationary Phase Decay (Morse Critical Point)')
plt.grid(True, which="both", ls=":")
plt.legend()
plt.tight_layout()
plt.show()


## Free Particle Propagator (Schrödinger Evolution)

In [ ]:
"""
semiclassical_free_gaussian_with_profile.py
"""


# ------------------------------------------------------------
# Physical parameters
# ------------------------------------------------------------

m = 1.0
t_val = 1.0
sigma = 1.0

# Observation point for convergence study
x_obs = 0.5

# ħ values for convergence plot
hbar_values = np.logspace(-2, -0.2, 80)

# ------------------------------------------------------------
# Symbolic setup
# ------------------------------------------------------------

x0 = sp.symbols('x0', real=True)
S_template = m * (sp.Symbol('x') - x0)**2 / (2 * t_val)
amp = sp.exp(-x0**2 / (4 * sigma**2))

evaluator = AsymptoticEvaluator()

# ------------------------------------------------------------
# Convergence study (error vs ħ)
# ------------------------------------------------------------

asym_vals = []
exact_vals = []

for hbar in hbar_values:

    lam = 1.0 / hbar

    # Phase at fixed observation point
    S = m * (x_obs - x0)**2 / (2 * t_val)
    analyzer = Analyzer(S, amp, [x0], domain=[(-10, 10)])
    points = analyzer.find_critical_points([np.array([x_obs])])
    cp = analyzer.analyze_point(points[0])

    res = evaluator.evaluate(cp, lam)

    prefactor = np.sqrt(m / (2j * np.pi * hbar * t_val))
    psi_asym = prefactor * res.total_value
    asym_vals.append(np.abs(psi_asym))

    # Exact
    sigma_t = sigma * np.sqrt(1 + (hbar * t_val / (2*m*sigma**2))**2)
    psi_exact = (2*np.pi*sigma_t**2)**(-0.25) * np.exp(-x_obs**2/(4*sigma_t**2))
    exact_vals.append(np.abs(psi_exact))

asym_vals = np.array(asym_vals)
exact_vals = np.array(exact_vals)

rel_error = np.abs((asym_vals - exact_vals) / exact_vals)


# ------------------------------------------------------------
# Profile comparison at small ħ
# ------------------------------------------------------------

hbar_small = 0.02
lam_small = 1.0 / hbar_small

x_vals = np.linspace(-3, 3, 300)
density_exact = []
density_asym = []

for xv in x_vals:

    S = m * (xv - x0)**2 / (2 * t_val)
    analyzer = Analyzer(S, amp, [x0], domain=[(-10, 10)])
    points = analyzer.find_critical_points([np.array([xv])])
    cp = analyzer.analyze_point(points[0])

    res = evaluator.evaluate(cp, lam_small)
    prefactor = np.sqrt(m / (2j * np.pi * hbar_small * t_val))
    psi_asym = prefactor * res.total_value

    density_asym.append(np.abs(psi_asym))

    sigma_t = sigma * np.sqrt(1 + (hbar_small * t_val / (2*m*sigma**2))**2)
    psi_exact = (2*np.pi*sigma_t**2)**(-0.25) * np.exp(-xv**2/(4*sigma_t**2))
    density_exact.append(np.abs(psi_exact))

density_exact = np.array(density_exact)
density_asym = np.array(density_asym)


# ------------------------------------------------------------
# Plotting
# ------------------------------------------------------------

plt.figure(figsize=(12,5))

# Error convergence
plt.subplot(1,2,1)
plt.loglog(hbar_values, rel_error, 'b-', linewidth=2)
plt.xlabel('ħ')
plt.ylabel('Relative error in |ψ|')
plt.title('Semiclassical Convergence (ħ → 0)')
plt.gca().invert_xaxis()
plt.grid(True, which="both", ls=":")

# Profile comparison
plt.subplot(1,2,2)
plt.plot(x_vals, density_exact, 'k-', linewidth=2.5, label='Exact')
plt.plot(x_vals, density_asym, 'r--', linewidth=2,
         label=f'Stationary phase (ħ={hbar_small})')

plt.xlabel('Position x')
plt.ylabel('|ψ(x,t)|')
plt.title('Wavefunction Profile Comparison')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()


## Pseudo-differential operator simultation with defined functions

In [ ]:
# --- Core PSDO Functions ---

def apply_psdo_1d(u_amp, u_phase, symbol_expr, domain, x_range, lambda_val=60.0):
    y, xi, x = sp.symbols('y xi x', real=True)
    # Construction de la phase totale du PSDO : (x-y)*xi + psi(y)
    total_phase = (x - y) * xi + u_phase.subs(list(u_phase.free_symbols)[0], y)
    total_amp = symbol_expr * u_amp.subs(list(u_amp.free_symbols)[0], y)
    
    evaluator = AsymptoticEvaluator()
    results = []
    
    for xv in x_range:
        # Substitution de la position d'observation x
        phi_x = total_phase.subs(x, xv)
        amp_x = total_amp.subs(x, xv)
        
        analyzer = Analyzer(phi_x, amp_x, [y, xi], domain=domain)
        # Guesses : x=y et xi=grad(psi)(x)
        guesses = [np.array([xv, 0.0]), np.array([xv, 1.0]), np.array([xv, -1.0])]
        points = analyzer.find_critical_points(guesses)
        
        val = 0j
        if points:
            for p in points:
                try:
                    cp = analyzer.analyze_point(p)
                    val += evaluator.evaluate(cp, lambda_val).total_value
                except: continue
        results.append((lambda_val / (2 * np.pi)) * val)
    return np.array(results)

# --- 1D Examples ---
x_sym, xi_sym, y_sym = sp.symbols('x xi y', real=True)
u_amp_1d = sp.exp(-y_sym**2)
u_phase_1d = y_sym**2 / 2
x_vals = np.linspace(-2.5, 2.5, 50)

examples_1d = [
    (sp.Integer(1), r"Identity: $S=1$"),
    (sp.exp(-xi_sym**2), r"Freq. Filter: $S=e^{-\xi^2}$"),
    (sp.exp(sp.I * xi_sym * 0.8), r"Translation: $S=e^{i 0.8 \xi}$"),
    (1 - 0.9 * sp.exp(-10 * x_sym**2), r"Spatial Mask: $1 - 0.9e^{-10x^2}$"),
    (sp.exp(sp.I * 0.3 * xi_sym**2), r"Dispersion: $S=e^{i 0.3 \xi^2}$")
]

plt.figure(figsize=(18, 5))
for i, (sym, title) in enumerate(examples_1d):
    res = apply_psdo_1d(u_amp_1d, u_phase_1d, sym, [(-5, 5), (-5, 5)], x_vals)
    plt.subplot(1, 5, i+1)
    plt.plot(x_vals, np.abs(res), 'b-', lw=2)
    plt.fill_between(x_vals, 0, np.abs(res), alpha=0.1, color='blue')
    plt.title(title, fontsize=14)
    plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def apply_psdo_2d(u_amp, u_phase, symbol_expr, domain, x_grid, y_grid, lambda_val=40.0):
    # Variables : y (espace), xi (fréquence), x (paramètre de sortie)
    y1, y2, xi1, xi2, x1, x2 = sp.symbols('y1 y2 xi1 xi2 x1 x2', real=True)
    
    # Phase totale : (x - y)·ξ + ψ(y)
    # On substitue les variables y dans la phase de l'onde initiale
    u_phase_y = u_phase.subs({list(u_phase.free_symbols)[0]: y1, list(u_phase.free_symbols)[1]: y2})
    total_phase = (x1 - y1)*xi1 + (x2 - y2)*xi2 + u_phase_y
    
    # Amplitude totale : S(x, ξ) * u_amp(y)
    u_amp_y = u_amp.subs({list(u_amp.free_symbols)[0]: y1, list(u_amp.free_symbols)[1]: y2})
    total_amp = symbol_expr * u_amp_y
    
    X1, X2 = np.meshgrid(x_grid, y_grid)
    results = np.zeros(X1.shape, dtype=complex)
    evaluator = AsymptoticEvaluator()
    
    # On itère sur la grille spatiale de sortie
    for i in range(X1.shape[0]):
        for j in range(X1.shape[1]):
            xv1, xv2 = X1[i, j], X2[i, j]
            
            # Localisation de la phase et de l'amplitude au point x
            phi_loc = total_phase.subs({x1: xv1, x2: xv2})
            amp_loc = total_amp.subs({x1: xv1, x2: xv2})
            
            analyzer = Analyzer(phi_loc, amp_loc, [y1, y2, xi1, xi2], domain=domain)
            
            # Guess théorique : y = x, xi = grad(psi)(x)
            # Pour psi = (y1**2 + y2**2)/2, grad = (y1, y2)
            guesses = [np.array([xv1, xv2, xv1, xv2])] 
            points = analyzer.find_critical_points(guesses)
            
            val = 0j
            for p in points:
                try:
                    cp = analyzer.analyze_point(p)
                    val += evaluator.evaluate(cp, lambda_val).total_value
                except: continue
            
            # Normalisation PSDO : (λ/2π)^n avec n=2
            results[i, j] = (lambda_val / (2 * np.pi))**2 * val
            
    return results

# --- Définition des exemples 2D ---
y1, y2, xi1, xi2, x1, x2 = sp.symbols('y1 y2 xi1 xi2 x1 x2', real=True)

# Onde initiale : Gaussienne avec un chirp quadratique
u_amp_2d = sp.exp(-(y1**2 + y2**2))
u_phase_2d = (y1**2 + y2**2) / 2

# Grille de calcul
x_grid = np.linspace(-1.5, 1.5, 20)
y_grid = np.linspace(-1.5, 1.5, 20)

examples_2d = [
    (sp.Integer(1), r"Identity"),
    (sp.exp(-2 * xi1**2), r"Low-pass $x_1$"),
    (sp.exp(-(x1**2 + x2**2)), r"Spatial Window"),
    (sp.exp(sp.I * xi1 * x2), r"Shear (Coupling $x_2, \xi_1$)"),
    (sp.exp(-(xi1**2 + 0.1 * xi2**2)), r"Anisotropic Filter")
]

# --- Visualisation ---
plt.figure(figsize=(20, 4))
for i, (sym, title) in enumerate(examples_2d):
    print(f"Calcul de l'exemple : {title}...")
    res = apply_psdo_2d(u_amp_2d, u_phase_2d, sym, [(-4, 4)]*4, x_grid, y_grid, lambda_val=40.0)
    
    plt.subplot(1, 5, i+1)
    # Affichage de la magnitude (module)
    im = plt.imshow(np.abs(res), extent=[-1.5, 1.5, -1.5, 1.5], 
                    cmap='magma', origin='lower', interpolation='bilinear')
    plt.title(title, fontsize=14)
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xlabel(r"$x_1$")
    if i == 0: plt.ylabel(r"$x_2$")

plt.tight_layout()
plt.show()

## Semi-Classical Wave Propagation

In [ ]:
# ============================================================
#  Semi-Classical Wave Propagation
#  Uses asymptotic.py (Analyzer + AsymptoticEvaluator)
#
#  For each PDE  ∂ₜu = L(∂ₓ)u  with dispersion relation ω(ξ),
#  the solution is expressed as a Fourier Integral Operator:
#
#      u(x,t) = (λ/2π) ∫∫ exp(iλ φ(x,y,ξ)) a(y,ξ) dy dξ
#
#  where  φ(x,y,ξ) = (x−y)ξ + t ω(ξ) + φ₀(y)
#  and    a(y,ξ)   = a₀(y) · S(ξ)
#
#  The dominant contributions come from stationary points of φ,
#  computed by asymptotic.py.  The FFT solution is overlaid
#  in grey as a silent numerical reference.
# ============================================================
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.integrate import quad, dblquad
from asymptotic import *

# Configuration pour de jolis graphiques
%matplotlib inline
plt.style.use('seaborn-v0_8-muted')
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.fft import fft, ifft, fftfreq, fftshift

# ── Colours ─────────────────────────────────────────────────
C_ASYMP = "#c0392b"   # red   – asymptotic (main result)
C_FFT   = "#aaaaaa"   # grey  – FFT reference (silent)
C_IC    = "#2980b9"   # blue  – initial condition


# ============================================================
#  Initial conditions
# ============================================================
# Each IC is a dict with keys:
#   sym   : SymPy expression in y   (enters the asymptotic amplitude)
#   np    : NumPy function of x     (used for FFT and plotting)
#   p0    : initial momentum        (enters as linear phase φ₀ = p0·y)
#   label : short human-readable description

def ic_gaussian(y_sym):
    """Pure Gaussian  u₀(x) = exp(−x²)"""
    return dict(
        sym   = sp.exp(-y_sym**2),
        np    = lambda x: np.exp(-x**2).astype(complex),
        p0    = 0.0,
        label = "u₀ = exp(−x²)",
    )

def ic_packet(y_sym, p0: float):
    """Gaussian wave packet  u₀(x) = exp(−x²)·exp(ip₀x)"""
    return dict(
        sym   = sp.exp(-y_sym**2),   # amplitude only; p0 enters via φ₀
        np    = lambda x: np.exp(-x**2) * np.exp(1j * p0 * x),
        p0    = p0,
        label = f"u₀ = exp(−x²)·exp(i·{p0}·x)",
    )

def ic_asymmetric(y_sym):
    """Asymmetric bump  u₀(x) = x·exp(−x²)"""
    return dict(
        sym   = y_sym * sp.exp(-y_sym**2),
        np    = lambda x: x * np.exp(-x**2).astype(complex),
        p0    = 0.0,
        label = "u₀ = x·exp(−x²)",
    )


# ============================================================
#  FFT reference propagator  (silent validation)
# ============================================================

def fft_propagate(ic_np, omega_np, x_arr, t, dual=False):
    """
    Exact spectral propagation:
        û(ξ,t) = û₀(ξ) · exp(it ω(ξ))     [single branch]
        û(ξ,t) = û₀(ξ) · cos(t ω(ξ))      [dual ±ω]
    Returns |u(x,t)|.
    """
    N      = len(x_arr)
    dx     = x_arr[1] - x_arr[0]
    u0_hat = fft(ic_np(x_arr))
    xi_arr = 2 * np.pi * fftfreq(N, d=dx)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        om = omega_np(xi_arr)
    u_hat = u0_hat * (np.cos(t * om) if dual else np.exp(1j * t * om))
    return np.abs(ifft(u_hat))


# ============================================================
#  Asymptotic propagator  (core of this example)
# ============================================================

def asymptotic_propagate(ic_sym, p0, omega_sym, amp_sym,
                         x_arr, t, lam, dual=False, is_laplace=False):
    """
    Evaluate u(x,t) at each point in x_arr via asymptotic.py.

    The phase of the FIO is:
        φ(y,ξ; x) = (x−y)·ξ + t·ω(ξ) + p0·y

    Analyzer finds the critical points (y_c, ξ_c) satisfying ∇φ = 0:
        ∂φ/∂y = 0  →  ξ_c = p0
        ∂φ/∂ξ = 0  →  y_c = x + t·ω'(ξ_c)

    AsymptoticEvaluator then returns the leading asymptotic contribution.

    Parameters
    ----------
    ic_sym     : SymPy expression for the initial amplitude a₀(y)
    p0         : initial momentum (shifts the ξ critical point)
    omega_sym  : SymPy dispersion relation ω(ξ)
    amp_sym    : SymPy spectral amplitude S(ξ)
    x_arr      : observation points
    t          : time
    lam        : large parameter λ
    dual       : True → sum ω and −ω branches (÷ 2)
    is_laplace : True → purely imaginary ω; use Laplace prefactor 1/(2π)
    """
    y, xi = sp.symbols('y xi', real=True)
    evaluator = AsymptoticEvaluator()
    phi0 = sp.Integer(0) if p0 == 0.0 else sp.Float(p0) * y

    results = []
    for xv in x_arr:
        total    = 0j
        branches = [omega_sym, -omega_sym] if dual else [omega_sym]

        for omega in branches:
            phase = (xv - y) * xi + t * omega + phi0
            amp   = ic_sym * amp_sym

            analyzer = Analyzer(phase, amp, [y, xi],
                                domain=[(-8, 8), (-8, 8)])

            # Seed the optimizer near the expected critical point
            guesses = [
                np.array([xv,        p0      ]),
                np.array([xv,        p0 + 1.0]),
                np.array([xv,        p0 - 1.0]),
                np.array([0.0,       p0      ]),
                np.array([xv * 0.5,  p0      ]),
                np.array([xv,        0.0     ]),
                np.array([xv,        2.0     ]),
                np.array([xv,       -2.0     ]),
            ]

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                points = analyzer.find_critical_points(guesses)

            for pt in points:
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")
                        cp  = analyzer.analyze_point(pt)
                        res = evaluator.evaluate(cp, lam)
                    total += res.total_value
                except Exception:
                    continue

        prefactor = 1.0 / (2 * np.pi) if is_laplace else lam / (2 * np.pi)
        results.append(prefactor * total / len(branches))

    return np.abs(np.array(results))


# ============================================================
#  Physics scenarios
# ============================================================

y_s  = sp.Symbol('y',  real=True)
xi_s = sp.Symbol('xi', real=True)

# Smooth |ξ| for the wave equation: avoids the non-differentiable
# point at ξ = 0 which causes the optimizer to diverge.
_EPS       = 0.12
_abs_xi    = sp.sqrt(xi_s**2 + _EPS**2)
_abs_xi_np = lambda xi: np.sqrt(xi**2 + _EPS**2)

scenarios = {

    "Stationary": dict(
        omega_sym  = sp.Integer(0),
        omega_np   = lambda xi: np.zeros_like(xi, dtype=complex),
        amp_sym    = sp.Integer(1),
        ic         = ic_gaussian(y_s),
        dual       = False,
        is_laplace = False,
        note       = "∂ₜu = 0\nu(x,t) = u₀(x)  [no evolution]",
    ),

    "Transport": dict(
        omega_sym  = -xi_s,
        omega_np   = lambda xi: -xi + 0j,
        amp_sym    = sp.Integer(1),
        ic         = ic_gaussian(y_s),
        dual       = False,
        is_laplace = False,
        note       = "∂ₜu + ∂ₓu = 0\ntranslation at speed 1",
    ),

    "Heat": dict(
        omega_sym  = sp.I * xi_s**2,
        omega_np   = lambda xi: 1j * xi**2,
        amp_sym    = sp.Integer(1),
        ic         = ic_gaussian(y_s),
        dual       = False,
        is_laplace = True,   # purely imaginary ω → Laplace saddle
        note       = "∂ₜu = ∂ₓₓu\nGaussian broadening  [Laplace saddle]",
    ),

    "Schrödinger": dict(
        omega_sym  = -xi_s**2 / 2,
        omega_np   = lambda xi: -xi**2 / 2 + 0j,
        amp_sym    = sp.Integer(1),
        ic         = ic_packet(y_s, p0=1.5),
        dual       = False,
        is_laplace = False,
        note       = "i∂ₜu = −½∂ₓₓu\nwave packet, vg = p₀ = 1.5",
    ),

    "Klein-Gordon": dict(
        omega_sym  = sp.sqrt(1 + xi_s**2),
        omega_np   = lambda xi: np.sqrt(1 + xi**2 + 0j),
        amp_sym    = sp.Integer(1),
        ic         = ic_gaussian(y_s),
        dual       = True,   # two branches ±ω
        is_laplace = False,
        note       = "(∂ₜₜ − ∂ₓₓ + 1)u = 0\nrelativistic dispersion  [dual ±ω]",
    ),

    "KdV": dict(
        omega_sym  = -xi_s**3,
        omega_np   = lambda xi: -xi**3 + 0j,
        amp_sym    = sp.Integer(1),
        ic         = ic_gaussian(y_s),
        dual       = False,
        is_laplace = False,
        note       = "∂ₜu + ∂ₓₓₓu = 0\ndispersive splitting",
    ),

    "Wave": dict(
        omega_sym  = _abs_xi,
        omega_np   = _abs_xi_np,
        amp_sym    = sp.Integer(1),
        ic         = ic_gaussian(y_s),
        dual       = True,   # two counter-propagating packets
        is_laplace = False,
        note       = "∂ₜₜu = ∂ₓₓu\ntwo counter-propagating packets  [dual ±ω]",
    ),

    "Biharmonic": dict(
        omega_sym  = -xi_s**4,
        omega_np   = lambda xi: -xi**4 + 0j,
        amp_sym    = sp.Integer(1),
        ic         = ic_asymmetric(y_s),
        dual       = False,
        is_laplace = False,
        note       = "∂ₜu + ∂ₓₓₓₓu = 0\nfast dispersive spreading",
    ),
}


# ============================================================
#  Parameters
# ============================================================
x_arr = np.linspace(-7, 7, 180)
t     = 0.5
LAM   = 60.0

print(f"Semi-Classical Propagation   t = {t}   λ = {LAM}")
print("=" * 52)


# ============================================================
#  Compute all scenarios
# ============================================================
results = {}

for name, sc in scenarios.items():
    print(f"  {name}…")
    ic = sc['ic']

    u_fft = fft_propagate(
        ic['np'], sc['omega_np'], x_arr, t, dual=sc['dual'])

    u_asymp = asymptotic_propagate(
        ic['sym'], ic['p0'],
        sc['omega_sym'], sc['amp_sym'],
        x_arr, t, LAM,
        dual=sc['dual'], is_laplace=sc['is_laplace'])

    results[name] = dict(
        u_fft   = u_fft,
        u_asymp = u_asymp,
        u_ic    = np.abs(ic['np'](x_arr)),
        ic_label= ic['label'],
        note    = sc['note'],
    )

print("Done.\n")


# ============================================================
#  Plot
# ============================================================
n  = len(scenarios)
nc = 4
nr = int(np.ceil(n / nc))

fig = plt.figure(figsize=(20, nr * 4.8))
fig.suptitle(
    f"Semi-Classical Wave Propagation   (t = {t},  λ = {LAM})\n"
    "Red = asymptotic.py     Grey = FFT reference     Blue dashed = initial condition",
    fontsize=13)

gs = gridspec.GridSpec(nr, nc, figure=fig, hspace=0.65, wspace=0.32)

for idx, (name, res) in enumerate(results.items()):
    ax = fig.add_subplot(gs[idx // nc, idx % nc])

    # Grey FFT reference — drawn first (background)
    ax.plot(x_arr, res['u_fft'],
            color=C_FFT, lw=1.4, ls='-', alpha=0.75, zorder=1)

    # Blue dashed initial condition
    ax.plot(x_arr, res['u_ic'],
            color=C_IC, lw=1.0, ls='--', alpha=0.55, zorder=2)

    # Red asymptotic result — main curve
    ax.fill_between(x_arr, 0, res['u_asymp'],
                    color=C_ASYMP, alpha=0.15, zorder=3)
    ax.plot(x_arr, res['u_asymp'],
            color=C_ASYMP, lw=2.4, zorder=4)

    ax.set_title(f"{name}\n{res['note']}", fontsize=8.5, pad=3)
    ax.set_xlabel("x", fontsize=9)
    ax.set_xlim(x_arr[0], x_arr[-1])
    ax.set_ylim(-0.05, 1.5)
    ax.grid(True, ls=':', alpha=0.4)

# Shared legend in first subplot
legend_handles = [
    Line2D([0], [0], color=C_ASYMP, lw=2.4,              label="asymptotic.py"),
    Line2D([0], [0], color=C_FFT,   lw=1.4,              label="FFT reference"),
    Line2D([0], [0], color=C_IC,    lw=1.0, ls='--',     label="initial cond."),
]
fig.axes[0].legend(handles=legend_handles, fontsize=7.5, loc='upper right')

# Hide unused subplots if n is not a multiple of nc
for extra in range(n, nr * nc):
    fig.axes[extra].set_visible(False)

plt.tight_layout()
plt.show()

## Contribution decomposition
### 1D, two Morse critical points (stationary phase)

In [ ]:
import numpy as np
import sympy as sp

x, y = sp.symbols('x y')
lams = np.logspace(1, 4, 80)

# ============================================================
# Snippet 1 — 1D, two Morse critical points (stationary phase)
# I(λ) = ∫ exp(iλ (x⁴/4 - x²/2)) dx
# Double-well phase: two symmetric stationary points at x = ±1.
# Demonstrates interference between the two contributions.
# ============================================================

phi_1 = x**4 / 4 - x**2 / 2          # stationary points at x = ±1
amp_1 = sp.Integer(1)

analyzer_1 = Analyzer(phi_1, amp_1, [x])
pts_1      = analyzer_1.find_critical_points([np.array([1.0]), np.array([-1.0])])
cps_1      = [analyzer_1.analyze_point(p) for p in pts_1]

fig1, axes1 = plot_contribution_decomposition(
    cps_1, analyzer_1,
    lambda_values=lams,
    show_correction=True,
    show_coherent_sum=True,
)


### 1D, Airy singularity (degenerate stationary phase)

In [ ]:
# ============================================================
# Snippet 2 — 1D, Airy singularity (degenerate stationary phase)
# I(λ) = ∫ exp(iλ (x³/3 + c·x)) dx  with c = 0 → cubic critical point
# The unique critical point at x = 0 is a corank-1 fold (Airy type).
# Correction term is unavailable; show_correction=False avoids the empty panel.
# ============================================================

phi_2 = x**3 / 3                      # degenerate at x = 0 (φ' = φ'' = 0)
amp_2 = 1 + x**2 / 4

analyzer_2 = Analyzer(phi_2, amp_2, [x])
pts_2      = analyzer_2.find_critical_points([np.array([0.0])])
cps_2      = [analyzer_2.analyze_point(p) for p in pts_2]

fig2, axes2 = plot_contribution_decomposition(
    cps_2, analyzer_2,
    lambda_values=lams,
    show_correction=False,   # Airy: correction not computed
    show_coherent_sum=False, # single point: coherent sum is redundant
)

### 2D, two Morse critical points (Laplace)

In [ ]:
# ============================================================
# Snippet 3 — 2D, two Morse critical points (Laplace)  [FIXED]
# I(λ) = ∫∫ exp(-λ ((x²-1)² + y²)) dx dy
# Two exponentially dominant minima at (±1, 0).
# ============================================================

phi_3 = (x**2 - 1)**2 + y**2          # real ψ — minima at (±1, 0)
amp_3 = sp.Integer(1)

# Pass phi_3 directly (real ψ) and force LAPLACE explicitly.
# Wrapping with sp.I was wrong: AUTO then detects SADDLE_POINT,
# the gradient solver operates in ℂ² and fails to find the real minima.
analyzer_3 = Analyzer(phi_3, amp_3, [x, y],
                      method=IntegralMethod.LAPLACE)

pts_3 = analyzer_3.find_critical_points([np.array([ 1.0, 0.0]),
                                         np.array([-1.0, 0.0])])
cps_3 = [analyzer_3.analyze_point(p) for p in pts_3]

fig3, axes3 = plot_contribution_decomposition(
    cps_3, analyzer_3,
    lambda_values=lams,
    show_correction=True,
    show_coherent_sum=True,
)

### 2D, single saddle point (steepest descent)

In [ ]:
# ============================================================
# Snippet 4 — 2D, single saddle point (steepest descent)
# I(λ) = ∫∫ exp(iλ ((1/2 + i/4)(x² + y²))) dx dy
# The phase is genuinely complex; AUTO detects SADDLE_POINT.
# The saddle lies at the origin in ℂ².
# ============================================================

phi_4 = (sp.Rational(1, 2) + sp.I * sp.Rational(1, 4)) * (x**2 + y**2)
amp_4 = sp.Integer(1)

analyzer_4  = Analyzer(phi_4, amp_4, [x, y])
sdl_eval_4  = SaddlePointEvaluator()
saddles_4   = sdl_eval_4.find_saddle_points(analyzer_4, [np.array([0.0, 0.0])])
cps_4       = [analyzer_4.analyze_point(s) for s in saddles_4]

fig4, axes4 = plot_contribution_decomposition(
    cps_4, analyzer_4,
    lambda_values=lams,
    show_correction=True,
    show_coherent_sum=False,  # single saddle: Panel 3 is redundant
)